# MLP 는 어떻게 계산하고, 학습으로 무엇이 바뀌는가

**「02. 딥러닝 개요」 실습 노트북 · v1.4 (핵심 개념 중심)**

---

## 00. 이번 실습에서 무엇을 볼까?

민원 문장 한 줄을 **3개 분야 중 하나로 분류하는 아주 작은 MLP** 를 직접 실행합니다.

| 들어가는 것 | 처리하는 것 | 나오는 것 |
|---|---|---|
| "소득금액증명서를 발급받고 싶습니다" | 작은 MLP | **증명서 발급** |

이 노트북에서 확인할 것은 **딱 두 가지 흐름**입니다.

| # | 흐름 | 내용 |
|---|---|---|
| **①** | **Forward (계산)** | 입력 → Dense → ReLU → Dense → Softmax → 예측 |
| **②** | **Learning (학습)** | 예측 → Loss → 역전파 → Optimizer → 학습 파라미터 수정 |

노트북을 다 본 뒤 이 두 흐름을 **자기 말로 설명할 수 있으면 성공**입니다.

---

## Python 을 몰라도 괜찮습니다

이 노트북의 목표는 Python 을 배우는 것이 **아닙니다.**
코드 셀은 **9개뿐**이고, 아래 표시만 보고 어디에 집중할지 정하면 됩니다.

| 표시 | 뜻 |
|---|---|
| `[입력 준비 — MLP 아님]` | 문장을 숫자로 바꾸는 준비 단계. **실행만 하고 넘어갑니다** |
| `★ [MLP 핵심]` | 오늘의 주제 ①. **집중해서 보세요** |
| `★ [딥러닝 학습 핵심]` | 오늘의 주제 ②. **집중해서 보세요** |
| `[결과 확인]` | 출력된 숫자만 눈으로 확인하면 됩니다 |
| `📍 현재 위치` | 지금 보고 있는 단계가 전체 흐름의 어디인지 알려 줍니다 |

> **회색 코드 상자는 전부 실제로 실행되는 Python 코드입니다.**
> 개념 설명은 그림과 표로만 되어 있으므로, 문법을 읽으려 애쓸 필요가 없습니다.

---

> ### 이 노트북은 모델 성능을 엄밀하게 측정하는 실습이 아닙니다.
>
> 이번 목적은 MLP 가 `입력 → 예측 → Loss → 역전파 → 값 수정` 과정을
> 어떻게 반복하는지 확인하는 것입니다.
> 그래서 정식 성능 평가(별도 시험 데이터 · 정확도 지표 비교)는 하지 않으며,
> 이 모델이 우수하다고 주장하지도 않습니다.

> **데이터 안내**
> 여기에 사용하는 민원 문장은 **전부 교육용으로 새로 지어낸 가상의 문장**입니다.
> 실제 민원 자료나 개인정보는 하나도 사용하지 않았습니다.

> **실행 방법** 위에서부터 셀을 순서대로 실행합니다. (`Shift + Enter`)

---
## 00-1. 오늘 볼 전체 흐름 한 장

아래 그림이 **이 노트북 전체의 지도**입니다.
지금 전부 이해하지 못해도 괜찮습니다. 뒤에서 한 구역씩 확대해서 다시 봅니다.

<svg viewBox="0 0 940 664" role="img" aria-label="노트북 전체 알고리즘 한 장" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>민원 분류 프로그램 전체 흐름</title><desc>민원 문장이 입력 준비 단계인 TextVectorization, Embedding, GlobalAveragePooling1D 를 거쳐 숫자 16개가 되고, MLP 핵심인 Dense 16 과 ReLU, Dense 3 과 Softmax 를 지나 3개 카테고리 점수가 되는 계산 흐름과, 그 점수를 정답과 비교해 Loss 를 구하고 역전파로 영향을 계산한 뒤 Optimizer 가 학습 파라미터를 실제로 수정하는 학습 반복 과정, 그리고 학습이 끝난 모델이 새 민원의 후보 카테고리를 제안하고 담당자가 최종 확인하는 실제 사용 과정을 다섯 구역으로 나누어 보여 주는 전체 지도.</desc><rect x="0" y="0" width="940" height="664" fill="#FFFFFF"/><text x="470" y="32" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">이 노트북 전체가 하는 일 한 장으로 보기</text><text x="470" y="56" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">회색 = 입력 준비 (MLP 아님)   ·   파랑 = ★ MLP 핵심   ·   주황 = ★ 학습</text><rect x="24" y="76" width="300" height="272" rx="14" ry="14" fill="#F7F8FA" stroke="#D7DBE2" stroke-width="1.5"/><text x="44" y="102" font-size="14" fill="#6B7280" text-anchor="start" font-weight="bold">① 입력 준비</text><text x="44" y="120" font-size="11.5" fill="#8B93A0" text-anchor="start" font-weight="normal">MLP 자체는 아닙니다</text><rect x="44" y="132" width="260" height="36" rx="8" ry="8" fill="#EDEFF3" stroke="#AEB6C2" stroke-width="1.5"/><text x="174" y="149" font-size="13" fill="#3D4552" text-anchor="middle" font-weight="bold">민원 문장</text><text x="174" y="162" font-size="9.5" fill="#6B7280" text-anchor="middle" font-weight="normal">한국어 한 줄</text><line x1="174" y1="168" x2="174" y2="173" stroke="#8A93A0" stroke-width="2"/><polygon points="174,182 168,172 180,172" fill="#8A93A0"/><rect x="44" y="182" width="260" height="36" rx="8" ry="8" fill="#EDEFF3" stroke="#AEB6C2" stroke-width="1.5"/><text x="174" y="199" font-size="13" fill="#3D4552" text-anchor="middle" font-weight="bold">TextVectorization</text><text x="174" y="212" font-size="9.5" fill="#6B7280" text-anchor="middle" font-weight="normal">글자 토큰 → 토큰 ID</text><line x1="174" y1="218" x2="174" y2="223" stroke="#8A93A0" stroke-width="2"/><polygon points="174,232 168,222 180,222" fill="#8A93A0"/><rect x="44" y="232" width="260" height="36" rx="8" ry="8" fill="#EDEFF3" stroke="#AEB6C2" stroke-width="1.5"/><text x="174" y="249" font-size="13" fill="#3D4552" text-anchor="middle" font-weight="bold">Embedding</text><text x="174" y="262" font-size="9.5" fill="#6B7280" text-anchor="middle" font-weight="normal">토큰 ID → 숫자 16개</text><line x1="174" y1="268" x2="174" y2="273" stroke="#8A93A0" stroke-width="2"/><polygon points="174,282 168,272 180,272" fill="#8A93A0"/><rect x="44" y="282" width="260" height="36" rx="8" ry="8" fill="#EDEFF3" stroke="#AEB6C2" stroke-width="1.5"/><text x="174" y="299" font-size="13" fill="#3D4552" text-anchor="middle" font-weight="bold">GlobalAveragePooling1D</text><text x="174" y="312" font-size="9.5" fill="#6B7280" text-anchor="middle" font-weight="normal">평균 → 문장 숫자 16개</text><text x="174" y="336" font-size="12.5" fill="#77400F" text-anchor="middle" font-weight="bold">→ 문장 하나 = 숫자 16개</text><rect x="356" y="76" width="300" height="272" rx="14" ry="14" fill="#F2F6FC" stroke="#4C78A8" stroke-width="2.5"/><text x="376" y="102" font-size="14" fill="#2F5C93" text-anchor="start" font-weight="bold">② ★ MLP 핵심</text><text x="376" y="120" font-size="11.5" fill="#4C78A8" text-anchor="start" font-weight="normal">오늘 이해해야 하는 부분</text><rect x="376" y="132" width="260" height="26" rx="8" ry="8" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="506" y="150" font-size="13" fill="#77400F" text-anchor="middle" font-weight="bold">입력 : 숫자 16개</text><line x1="506" y1="158" x2="506" y2="163" stroke="#8A93A0" stroke-width="2"/><polygon points="506,172 500,162 512,162" fill="#8A93A0"/><rect x="376" y="172" width="260" height="42" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="2"/><text x="506" y="190" font-size="14" fill="#1B3A5E" text-anchor="middle" font-weight="bold">Dense(16)</text><text x="506" y="206" font-size="11" fill="#1B3A5E" text-anchor="middle" font-weight="normal">Weight × Input + Bias</text><line x1="506" y1="214" x2="506" y2="219" stroke="#8A93A0" stroke-width="2"/><polygon points="506,228 500,218 512,218" fill="#8A93A0"/><rect x="376" y="228" width="260" height="24" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="506" y="245" font-size="13" fill="#0E4A46" text-anchor="middle" font-weight="bold">ReLU</text><line x1="506" y1="252" x2="506" y2="257" stroke="#8A93A0" stroke-width="2"/><polygon points="506,266 500,256 512,256" fill="#8A93A0"/><rect x="376" y="266" width="260" height="42" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="2"/><text x="506" y="284" font-size="14" fill="#1B3A5E" text-anchor="middle" font-weight="bold">Dense(3)</text><text x="506" y="300" font-size="11" fill="#1B3A5E" text-anchor="middle" font-weight="normal">Weight × Hidden + Bias</text><line x1="506" y1="308" x2="506" y2="313" stroke="#8A93A0" stroke-width="2"/><polygon points="506,322 500,312 512,312" fill="#8A93A0"/><rect x="376" y="322" width="260" height="24" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="506" y="339" font-size="13" fill="#0E4A46" text-anchor="middle" font-weight="bold">Softmax</text><rect x="688" y="76" width="228" height="272" rx="14" ry="14" fill="#F7F8FA" stroke="#D7DBE2" stroke-width="1.5"/><text x="708" y="102" font-size="14" fill="#1D4726" text-anchor="start" font-weight="bold">③ 결과</text><rect x="708" y="132" width="188" height="44" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="2"/><text x="802" y="160" font-size="14" fill="#1D4726" text-anchor="middle" font-weight="bold">3개 카테고리 점수</text><rect x="708" y="192" width="188" height="24" rx="6" ry="6" fill="#FFFFFF" stroke="#4E9A57" stroke-width="1.2"/><text x="802" y="209" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="normal">증명서 발급</text><rect x="708" y="220" width="188" height="24" rx="6" ry="6" fill="#FFFFFF" stroke="#4E9A57" stroke-width="1.2"/><text x="802" y="237" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="normal">신고·납부</text><rect x="708" y="248" width="188" height="24" rx="6" ry="6" fill="#FFFFFF" stroke="#4E9A57" stroke-width="1.2"/><text x="802" y="265" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="normal">홈택스 이용</text><line x1="802" y1="272" x2="802" y2="281" stroke="#8A93A0" stroke-width="2"/><polygon points="802,290 796,280 808,280" fill="#8A93A0"/><rect x="708" y="290" width="188" height="44" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="802" y="308" font-size="12" fill="#1D4726" text-anchor="middle" font-weight="normal">가장 높은 카테고리를</text><text x="802" y="325" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="bold">후보로 제안</text><line x1="328" y1="212" x2="345" y2="212" stroke="#8A93A0" stroke-width="2"/><polygon points="354,212 344,206 344,218" fill="#8A93A0"/><line x1="660" y1="212" x2="677" y2="212" stroke="#8A93A0" stroke-width="2"/><polygon points="686,212 676,206 676,218" fill="#8A93A0"/><rect x="24" y="386" width="560" height="210" rx="14" ry="14" fill="#FDF6EE" stroke="#DE8A3E" stroke-width="2.5"/><text x="44" y="412" font-size="14" fill="#77400F" text-anchor="start" font-weight="bold">④ ★ 학습  —  model.fit()</text><rect x="44" y="424" width="92" height="68" rx="8" ry="8" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="59" cy="439" r="11" fill="#7B5EA7"/><text x="59" y="444" font-size="12" fill="#FFFFFF" text-anchor="middle" font-weight="bold">1</text><text x="90" y="468" font-size="13" fill="#3B295D" text-anchor="middle" font-weight="bold">예측</text><text x="90" y="484" font-size="10.5" fill="#3B295D" text-anchor="middle" font-weight="normal">지금 값으로 계산</text><line x1="138" y1="458" x2="141" y2="458" stroke="#8A93A0" stroke-width="2"/><polygon points="150,458 140,452 140,464" fill="#8A93A0"/><rect x="151" y="424" width="92" height="68" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><circle cx="166" cy="439" r="11" fill="#4C78A8"/><text x="166" y="444" font-size="12" fill="#FFFFFF" text-anchor="middle" font-weight="bold">2</text><text x="197" y="468" font-size="13" fill="#1B3A5E" text-anchor="middle" font-weight="bold">정답과 비교</text><text x="197" y="484" font-size="10.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">Loss 계산</text><line x1="245" y1="458" x2="248" y2="458" stroke="#8A93A0" stroke-width="2"/><polygon points="257,458 247,452 247,464" fill="#8A93A0"/><rect x="258" y="424" width="92" height="68" rx="8" ry="8" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="273" cy="439" r="11" fill="#7B5EA7"/><text x="273" y="444" font-size="12" fill="#FFFFFF" text-anchor="middle" font-weight="bold">3</text><text x="304" y="468" font-size="13" fill="#3B295D" text-anchor="middle" font-weight="bold">역전파</text><text x="304" y="484" font-size="10.5" fill="#3B295D" text-anchor="middle" font-weight="normal">영향만 계산</text><line x1="352" y1="458" x2="355" y2="458" stroke="#8A93A0" stroke-width="2"/><polygon points="364,458 354,452 354,464" fill="#8A93A0"/><rect x="365" y="424" width="92" height="68" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><circle cx="380" cy="439" r="11" fill="#4E9A57"/><text x="380" y="444" font-size="12" fill="#FFFFFF" text-anchor="middle" font-weight="bold">4</text><text x="411" y="468" font-size="13" fill="#1D4726" text-anchor="middle" font-weight="bold">Optimizer</text><text x="411" y="484" font-size="10.5" fill="#1D4726" text-anchor="middle" font-weight="normal">값을 실제 수정</text><line x1="459" y1="458" x2="462" y2="458" stroke="#8A93A0" stroke-width="2"/><polygon points="471,458 461,452 461,464" fill="#8A93A0"/><rect x="472" y="424" width="92" height="68" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><circle cx="487" cy="439" r="11" fill="#4E9A57"/><text x="487" y="444" font-size="12" fill="#FFFFFF" text-anchor="middle" font-weight="bold">5</text><text x="518" y="468" font-size="13" fill="#1D4726" text-anchor="middle" font-weight="bold">파라미터</text><text x="518" y="484" font-size="10.5" fill="#1D4726" text-anchor="middle" font-weight="normal">수정 완료</text><line x1="518" y1="492" x2="518" y2="540" stroke="#8A93A0" stroke-width="2"/><line x1="518" y1="540" x2="90" y2="540" stroke="#8A93A0" stroke-width="2"/><line x1="90" y1="540" x2="90" y2="501" stroke="#8A93A0" stroke-width="2"/><polygon points="90,492 84,502 96,502" fill="#8A93A0"/><text x="304" y="532" font-size="12" fill="#77400F" text-anchor="middle" font-weight="bold">↺ 이 한 바퀴가 계속 반복됩니다</text><rect x="616" y="386" width="300" height="210" rx="14" ry="14" fill="#F7F8FA" stroke="#D7DBE2" stroke-width="1.5"/><text x="636" y="412" font-size="13.5" fill="#3D4552" text-anchor="start" font-weight="bold">⑤ 실제 사용 (학습이 끝난 뒤)</text><rect x="636" y="424" width="260" height="28" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="766" y="443" font-size="12.5" fill="#1B3A5E" text-anchor="middle" font-weight="bold">새 민원 문장</text><line x1="766" y1="452" x2="766" y2="457" stroke="#8A93A0" stroke-width="2"/><polygon points="766,466 760,456 772,456" fill="#8A93A0"/><rect x="636" y="466" width="260" height="28" rx="8" ry="8" fill="#EDEFF3" stroke="#AEB6C2" stroke-width="1.5"/><text x="766" y="485" font-size="12.5" fill="#3D4552" text-anchor="middle" font-weight="bold">학습된 모델 — 계산만</text><line x1="766" y1="494" x2="766" y2="499" stroke="#8A93A0" stroke-width="2"/><polygon points="766,508 760,498 772,498" fill="#8A93A0"/><rect x="636" y="508" width="260" height="28" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="766" y="527" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="bold">3개 점수 → 가장 높은 후보</text><line x1="766" y1="536" x2="766" y2="541" stroke="#8A93A0" stroke-width="2"/><polygon points="766,550 760,540 772,540" fill="#8A93A0"/><rect x="636" y="550" width="260" height="28" rx="8" ry="8" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="766" y="569" font-size="12.5" fill="#77400F" text-anchor="middle" font-weight="bold">담당자 최종 확인</text><line x1="802" y1="348" x2="802" y2="368" stroke="#8A93A0" stroke-width="2" stroke-dasharray="5 4"/><line x1="802" y1="368" x2="304" y2="368" stroke="#8A93A0" stroke-width="2" stroke-dasharray="5 4"/><line x1="304" y1="368" x2="304" y2="377" stroke="#8A93A0" stroke-width="2"/><polygon points="304,386 298,376 310,376" fill="#8A93A0"/><rect x="382" y="354" width="342" height="20" fill="#FFFFFF"/><text x="553" y="369" font-size="12" fill="#77400F" text-anchor="middle" font-weight="bold">학습할 때는 이 점수를 정답과 비교합니다</text><line x1="588" y1="491" x2="605" y2="491" stroke="#8A93A0" stroke-width="2"/><polygon points="614,491 604,485 604,497" fill="#8A93A0"/><text x="470" y="624" font-size="13.5" fill="#3C4552" text-anchor="middle" font-weight="normal">① 입력 준비는 MLP 가 아닙니다. MLP 는 숫자 16개를 받는 ② 부터 시작합니다.</text><text x="470" y="648" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">④ 학습은 모델의 구조를 바꾸는 것이 아니라, 모델 내부의 학습 가능한 값을 조정하는 과정입니다.</text></svg>

### 그림을 다섯 덩어리로 읽으세요

| 구역 | 무엇을 하나 | MLP 인가? |
|---|---|---|
| **① 입력 준비** | 민원 문장을 숫자 16개로 바꿉니다 | **아닙니다** |
| **② ★ MLP 핵심** | 숫자 16개를 3개 점수로 바꿉니다 | **네, 오늘의 주제** |
| **③ 결과** | 3개 점수 중 가장 높은 것을 후보로 제안합니다 | 출력 |
| **④ ★ 학습** | 점수를 정답과 비교해 내부 값을 고칩니다 | 학습 과정 |
| **⑤ 실제 사용** | 학습이 끝난 모델로 새 민원을 분류합니다 | 추론 |

### 여기서 딱 하나만 기억한다면

> **TextVectorization · Embedding · Pooling 은 MLP 가 아닙니다.**
> 문장을 MLP 가 계산할 수 있는 **숫자 16개**로 만들어 주는 **준비 단계**입니다.
> MLP 가 실제로 일을 시작하는 지점은 그 **숫자 16개**부터입니다.

---
## `[입력 준비 — MLP 아님]`  도구 불러오기

실습에 필요한 도구는 **두 개뿐**입니다. 오류 없이 실행되면 됩니다.

In [1]:
# ============================================================
# [입력 준비 — MLP 알고리즘이 아닙니다]
#
# numpy      : 숫자 목록을 다루는 도구
# tensorflow : 신경망을 만들고 학습시키는 도구
#
# 이 셀은 도구를 불러오기만 합니다. 계산은 아직 하지 않습니다.
# ============================================================
import numpy as np
import tensorflow as tf

tf.get_logger().setLevel("ERROR")

# 실행할 때마다 같은 결과가 나오도록 난수를 고정합니다.
tf.keras.utils.set_random_seed(42)

print("TensorFlow 버전 :", tf.__version__)

TensorFlow 버전 : 2.21.0


---
---
# 01. `[입력 준비 — MLP 아님]`  민원 문장과 정답

모델이 배울 내용은 **딱 이 관계 하나**입니다.

| 민원 문장 | 정답 |
|---|---|
| 소득금액증명서를 발급받고 싶습니다 | 증명서 발급 |
| 종합소득세 신고 방법을 알려주세요 | 신고·납부 |
| 홈택스 로그인이 되지 않습니다 | 홈택스 이용 |

### 분류할 3개 카테고리

| 번호 | 카테고리 | 어떤 문의인가 |
|---|---|---|
| 0 | **증명서 발급** | 발급 · 출력 · 증명 · 서류 · 증명원 |
| 1 | **신고·납부** | 신고 · 납부 · 세금 · 신고기한 |
| 2 | **홈택스 이용** | 로그인 · 인증 · 오류 · 접속 |

딥러닝은 규칙을 사람이 적어 주는 방식이 아니라 **예시를 보고 규칙을 스스로 찾는** 방식입니다.
그래서 이런 짝이 필요합니다. 이 실습에서는 **카테고리마다 18문장씩, 모두 54문장**을 씁니다.

> 아래 셀의 Python 리스트를 **한 줄씩 읽거나 외울 필요는 전혀 없습니다.**
> 실행만 하고 넘어가면 됩니다.

In [2]:
# ============================================================
# [실습용 데이터 준비 — MLP 알고리즘이 아닙니다]
#
# 아래 문장들은 모델이 학습할 교육용 가상 민원입니다.
# 이 Python 리스트를 한 줄씩 읽거나 외울 필요는 없습니다.
#
# 중요한 것은:
# 민원 문장 → 정답 카테고리
# 관계가 있다는 점뿐입니다.
# ============================================================

증명서발급 = [
    "소득금액증명서를 발급받고 싶습니다",
    "사업자등록증명원은 어디에서 받을 수 있나요",
    "납세증명서를 출력하려면 어떻게 해야 하나요",
    "부가가치세 과세표준증명을 신청하고 싶어요",
    "증명서를 떼려면 어떤 절차가 필요한가요",
    "소득 증명 서류 발급 방법이 궁금합니다",
    "폐업사실증명원 발급 신청은 어디에서 하나요",
    "영문 납세증명서도 발급이 되나요",
    "증명 서류를 집에서 인쇄할 수 있나요",
    "표준재무제표증명 발급 절차를 알려주세요",
    "세무서에 방문해서 증명서를 받을 수 있나요",
    "증명서 발급 수수료가 있는지 궁금합니다",
    "발급받은 증명서의 유효기간은 얼마인가요",
    "근로소득 원천징수영수증을 받고 싶어요",
    "소득금액증명원을 온라인으로 뽑을 수 있나요",
    "증명서를 파일로 저장하고 싶습니다",
    "국세납세증명서를 관공서에 제출하라고 합니다",
    "과거 연도 소득 증명 자료도 뽑을 수 있나요",
]

신고납부 = [
    "종합소득세 신고 방법을 알려주세요",
    "부가가치세 납부는 어떻게 하나요",
    "종합소득세 신고 기한이 언제까지인가요",
    "세금을 카드로 납부할 수 있나요",
    "신고 기한을 넘기면 가산세가 붙나요",
    "납부할 세액을 나누어 낼 수 있나요",
    "종합소득세를 잘못 신고했는데 수정할 수 있나요",
    "원천세 신고는 매달 해야 하나요",
    "환급금은 언제 받을 수 있나요",
    "세금 납부 계좌번호를 알고 싶습니다",
    "프리랜서도 종합소득세를 신고해야 하나요",
    "간이과세자 부가가치세 신고 방법이 궁금합니다",
    "납부 기한 연장을 신청할 수 있나요",
    "양도소득세는 언제까지 신고하나요",
    "이미 낸 세금을 돌려받으려면 어떻게 하나요",
    "무실적이어도 신고를 해야 하나요",
    "가산세 계산 방법을 알려주세요",
    "미납 세금이 있는지 확인하고 싶습니다",
]

홈택스이용 = [
    "홈택스 로그인이 되지 않습니다",
    "홈택스 비밀번호를 잊어버렸습니다",
    "홈택스에 접속하면 오류 화면이 뜹니다",
    "공동인증서가 홈택스에서 인식되지 않습니다",
    "홈택스 회원가입은 어떻게 하나요",
    "홈택스 아이디를 찾고 싶습니다",
    "손택스 앱 설치 방법을 알려주세요",
    "홈택스 화면이 계속 멈춥니다",
    "간편인증으로 홈택스에 들어갈 수 있나요",
    "홈택스에서 메뉴를 찾을 수가 없습니다",
    "인증서 갱신 후 홈택스 이용이 안 됩니다",
    "홈택스 사이트가 열리지 않습니다",
    "로그인 시도 횟수 초과로 계정이 잠겼습니다",
    "브라우저에서 홈택스 화면이 깨져 보입니다",
    "홈택스에서 파일 첨부가 되지 않습니다",
    "홈택스에서 자꾸 로그아웃이 됩니다",
    "홈택스 접속 시 프로그램 설치를 요구합니다",
    "컴퓨터를 바꾼 뒤 홈택스 인증이 안 됩니다",
]

CLASS_NAMES = ["증명서 발급", "신고·납부", "홈택스 이용"]

민원_문장 = 증명서발급 + 신고납부 + 홈택스이용
정답_번호 = [0] * len(증명서발급) + [1] * len(신고납부) + [2] * len(홈택스이용)

X = np.array(민원_문장, dtype=object)   # 민원 문장
y = np.array(정답_번호)                 # 정답 카테고리 번호

# 세 카테고리가 골고루 섞이도록 순서를 한 번 섞습니다.
섞은순서 = np.random.RandomState(42).permutation(len(X))
X = X[섞은순서]
y = y[섞은순서]

print("전체 민원 문장 :", len(X), "건")

전체 민원 문장 : 54 건


---
---
# 02. `[입력 준비 — MLP 아님]`  입력 준비

## 02-1. 문장을 **글자 토큰**으로 나누기

MLP 는 **숫자만** 계산할 수 있습니다. 한국어 문장은 그대로 넣을 수 없습니다.
그래서 문장을 먼저 **작은 조각(토큰)** 으로 나눕니다.

### 먼저 두 낱말만 알고 갑시다

| 낱말 | 쉬운 뜻 |
|---|---|
| **토큰 (Token)** | 문장을 모델이 처리하기 위해 나눈 **작은 단위** |
| **토큰 ID (Token ID)** | 각 토큰을 구분하기 위해 붙인 **번호** |

### 이번 실습의 토큰은 **글자**입니다

| | |
|---|---|
| 민원 문장 | "소득금액증명서를 발급받고 싶습니다" |
| **글자 토큰** | 소 / 득 / 금 / 액 / 증 / 명 / 서 / 를 / … |

> **이번 실습에서는 설명과 구현을 단순하게 유지하기 위해 글자 단위 토큰을 사용합니다.**
> 코드에서는 `split="character"` 한 줄이 이 역할을 합니다.

### 왜 글자 단위인가요?

실제 자연어 처리에서는 **글자 · 어절 · 서브워드** 등 다양한 단위로 문장을 나눌 수 있습니다.
이번 실습은 **MLP 의 동작 원리를 쉽게 확인하는 것이 목적**이므로 글자 단위를 사용합니다.

글자 단위는 한국어 조사 변화("증명서**를**", "증명서**가**")에도 덜 민감합니다.

> 한국어에서 공백으로 나눈 조각이 항상 "단어"와 같은 것은 아닙니다.
> 그래서 이 노트북에서는 "단어"라는 말 대신 **토큰**이라는 말을 씁니다.

---
## 02-2. 글자 토큰에 **토큰 ID** 붙이기

### 지금 무엇이 들어오나요?
민원 문장 한 줄입니다.

### 이 단계에서 무엇을 하나요?
문장을 글자 토큰으로 자르고, 사전을 보고 **토큰마다 정해진 번호(토큰 ID)** 를 붙입니다.

### 출력은 무엇인가요?
**토큰 ID 가 나열된 숫자 배열**입니다. (길이 45 로 맞춥니다)

<svg viewBox="0 0 940 512" role="img" aria-label="민원 문장이 글자 토큰과 토큰 ID 로 바뀌는 과정" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:880px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>민원 문장이 글자 토큰 ID 로 바뀌는 과정</title><desc>민원 문장을 글자 단위 토큰으로 자른 뒤 각 토큰에 사전의 토큰 ID 를 붙여 길이 45 의 정수 배열로 바꾸는 과정을 나타낸 그림. 토큰 ID 의 크기에는 의미가 없고 같은 글자에 같은 번호를 붙인 것이라는 설명을 포함한다.</desc><rect x="0" y="0" width="940" height="512" fill="#FFFFFF"/><text x="470" y="32" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">민원 문장 → 글자 토큰 → 토큰 ID</text><text x="470" y="56" font-size="13" fill="#77400F" text-anchor="middle" font-weight="bold">이번 실습의 토큰 단위 : 글자 (split="character")</text><rect x="250" y="74" width="440" height="62" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="470" y="96" font-size="13" fill="#1B3A5E" text-anchor="middle" font-weight="normal" opacity="0.85">민원 문장</text><text x="470" y="122" font-size="16" fill="#1B3A5E" text-anchor="middle" font-weight="bold">“소득금액증명서를 발급받고 싶습니다”</text><line x1="470" y1="136" x2="470" y2="155" stroke="#8A93A0" stroke-width="2"/><polygon points="470,164 464,154 476,154" fill="#8A93A0"/><text x="20" y="208" font-size="12" fill="#6B7280" text-anchor="start" font-weight="normal">① 글자로 자르기</text><text x="20" y="292" font-size="12" fill="#6B7280" text-anchor="start" font-weight="normal">② 토큰 ID 붙이기</text><rect x="142" y="174" width="64" height="54" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="174" y="209" font-size="24" fill="#0E4A46" text-anchor="middle" font-weight="bold">소</text><line x1="174" y1="228" x2="174" y2="262" stroke="#8A93A0" stroke-width="1.5" stroke-dasharray="3 3"/><rect x="142" y="262" width="64" height="46" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="174" y="292" font-size="19" fill="#1B3A5E" text-anchor="middle" font-weight="bold">27</text><rect x="216" y="174" width="64" height="54" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="248" y="209" font-size="24" fill="#0E4A46" text-anchor="middle" font-weight="bold">득</text><line x1="248" y1="228" x2="248" y2="262" stroke="#8A93A0" stroke-width="1.5" stroke-dasharray="3 3"/><rect x="216" y="262" width="64" height="46" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="248" y="292" font-size="19" fill="#1B3A5E" text-anchor="middle" font-weight="bold">29</text><rect x="290" y="174" width="64" height="54" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="322" y="209" font-size="24" fill="#0E4A46" text-anchor="middle" font-weight="bold">금</text><line x1="322" y1="228" x2="322" y2="262" stroke="#8A93A0" stroke-width="1.5" stroke-dasharray="3 3"/><rect x="290" y="262" width="64" height="46" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="322" y="292" font-size="19" fill="#1B3A5E" text-anchor="middle" font-weight="bold">30</text><rect x="364" y="174" width="64" height="54" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="396" y="209" font-size="24" fill="#0E4A46" text-anchor="middle" font-weight="bold">액</text><line x1="396" y1="228" x2="396" y2="262" stroke="#8A93A0" stroke-width="1.5" stroke-dasharray="3 3"/><rect x="364" y="262" width="64" height="46" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="396" y="292" font-size="19" fill="#1B3A5E" text-anchor="middle" font-weight="bold">70</text><rect x="438" y="174" width="64" height="54" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="470" y="209" font-size="24" fill="#0E4A46" text-anchor="middle" font-weight="bold">증</text><line x1="470" y1="228" x2="470" y2="262" stroke="#8A93A0" stroke-width="1.5" stroke-dasharray="3 3"/><rect x="438" y="262" width="64" height="46" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="470" y="292" font-size="19" fill="#1B3A5E" text-anchor="middle" font-weight="bold">9</text><rect x="512" y="174" width="64" height="54" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="544" y="209" font-size="24" fill="#0E4A46" text-anchor="middle" font-weight="bold">명</text><line x1="544" y1="228" x2="544" y2="262" stroke="#8A93A0" stroke-width="1.5" stroke-dasharray="3 3"/><rect x="512" y="262" width="64" height="46" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="544" y="292" font-size="19" fill="#1B3A5E" text-anchor="middle" font-weight="bold">16</text><rect x="586" y="174" width="64" height="54" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="618" y="209" font-size="24" fill="#0E4A46" text-anchor="middle" font-weight="bold">서</text><line x1="618" y1="228" x2="618" y2="262" stroke="#8A93A0" stroke-width="1.5" stroke-dasharray="3 3"/><rect x="586" y="262" width="64" height="46" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="618" y="292" font-size="19" fill="#1B3A5E" text-anchor="middle" font-weight="bold">5</text><rect x="660" y="174" width="64" height="54" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="692" y="209" font-size="24" fill="#0E4A46" text-anchor="middle" font-weight="bold">를</text><line x1="692" y1="228" x2="692" y2="262" stroke="#8A93A0" stroke-width="1.5" stroke-dasharray="3 3"/><rect x="660" y="262" width="64" height="46" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="692" y="292" font-size="19" fill="#1B3A5E" text-anchor="middle" font-weight="bold">17</text><rect x="734" y="174" width="64" height="54" rx="8" ry="8" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="766" y="209" font-size="20" fill="#555C68" text-anchor="middle" font-weight="bold">…</text><rect x="734" y="262" width="64" height="46" rx="8" ry="8" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="766" y="292" font-size="17" fill="#555C68" text-anchor="middle" font-weight="bold">…</text><line x1="470" y1="308" x2="470" y2="327" stroke="#8A93A0" stroke-width="2"/><polygon points="470,336 464,326 476,326" fill="#8A93A0"/><rect x="250" y="336" width="440" height="70" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="470" y="364" font-size="16" fill="#1D4726" text-anchor="middle" font-weight="bold">신경망이 계산할 수 있는 숫자 배열</text><text x="470" y="390" font-size="13.5" fill="#1D4726" text-anchor="middle" font-weight="normal">[ 27  29  30  70   9  16  5  17  …  0  0 ]   ← 길이 45</text><rect x="150" y="424" width="640" height="62" rx="10" ry="10" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="470" y="450" font-size="14.5" fill="#77400F" text-anchor="middle" font-weight="normal">토큰 ID 의 크기 자체에 의미가 있는 것이 아니라,</text><text x="470" y="473" font-size="14.5" fill="#77400F" text-anchor="middle" font-weight="bold">같은 글자에는 같은 번호를 붙인 것입니다. (30 이 27 보다 크다는 뜻이 아닙니다)</text></svg>

### 그림에서 확인할 점
- 문장이 **한 글자씩** 잘리고, 글자마다 **정해진 토큰 ID** 가 붙습니다.
- 토큰 ID 의 **크기 자체에는 의미가 없습니다.** "같은 글자에는 같은 번호" 라는 약속일 뿐입니다.
  `소 → 27`, `금 → 30` 이라고 해서 **`금` 이 `소` 보다 더 중요하다는 뜻이 아닙니다.**
- 문장이 짧으면 남는 자리는 **0(빈칸)** 으로 채워 길이를 맞춥니다.

### 왜 필요한가요?
신경망은 글자를 그대로 읽지 못합니다. **숫자로 바꿔 주지 않으면 계산 자체를 시작할 수 없습니다.**

### 다음 단계에는 무엇이 넘어가나요?
이 **토큰 ID** 들이 **Embedding** 으로 넘어갑니다.

In [3]:
# ============================================================
# [입력 준비 — MLP 알고리즘이 아닙니다]
#
# 입력 : 민원 문장 (글자)
# 처리 : 문장을 글자 토큰으로 자르고, 토큰마다 토큰 ID 를 붙입니다.
# 출력 : 길이 45 의 토큰 ID 배열
#
# → 다음 단계인 Embedding 으로 넘어갑니다.
# ============================================================
vectorize = tf.keras.layers.TextVectorization(
    max_tokens=2000,             # 기억할 토큰(글자) 종류의 최대 개수
    output_sequence_length=45,   # 길이를 45로 맞춥니다. 짧으면 0(빈칸)으로 채웁니다.
    split="character",           # ★ 이번 실습의 토큰 단위 = 글자
)

# adapt : 우리 민원 문장에 어떤 글자가 있는지 훑어보고 토큰 사전을 만듭니다.
vectorize.adapt(X)

print("준비 완료 : 이제 민원 문장을 토큰 ID 로 바꿀 수 있습니다.")

준비 완료 : 이제 민원 문장을 토큰 ID 로 바꿀 수 있습니다.


---
## 02-3. 준비 ② **Embedding**

> **📍 현재 위치**  
> 민원 문장 → 글자 토큰 → 토큰 ID → **★ Embedding ← 지금 여기** → Pooling → Dense → ReLU → Dense → Softmax

### Embedding 이란?

> **Embedding 은 토큰 ID 하나를 학습 가능한 숫자 벡터로 바꾸는 층입니다.**

토큰은 모델 설계에 따라 **글자 · 어절 · 단어 · 서브워드** 무엇이든 될 수 있습니다.
따라서 Embedding 을 "단어를 벡터로 바꾸는 층" 이라고 단정하면 정확하지 않습니다.
**이번 실습에서는 토큰이 글자이므로, 글자 하나마다 숫자 16개가 만들어집니다.**

### 지금 무엇이 들어오나요?
앞 단계가 만든 **토큰 ID** 입니다.

### 이 단계에서 무엇을 하나요?
토큰 ID 하나를 **숫자 16개의 묶음**으로 바꿉니다.

<svg viewBox="0 0 940 452" role="img" aria-label="글자 토큰 하나가 숫자 16개로 바뀌는 과정" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:880px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>Embedding : 토큰 ID → 숫자 16개</title><desc>글자 토큰 소, 득, 금 이 각각 토큰 ID 27, 29, 30 을 거쳐 Embedding 층에서 서로 다른 숫자 16개의 묶음으로 바뀌는 과정을 보여 주는 그림. 토큰 ID 는 식별 번호일 뿐이고 Embedding 의 숫자는 학습으로 조정된다는 설명을 포함한다.</desc><rect x="0" y="0" width="940" height="452" fill="#FFFFFF"/><text x="470" y="32" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">Embedding : 토큰 ID 하나 → 숫자 16개</text><text x="470" y="56" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">이번 실습은 토큰이 글자이므로, 글자 하나마다 숫자 16개가 만들어집니다.</text><text x="70" y="88" font-size="12" fill="#6B7280" text-anchor="middle" font-weight="bold">글자 토큰</text><text x="166" y="88" font-size="12" fill="#6B7280" text-anchor="middle" font-weight="bold">토큰 ID</text><text x="305" y="88" font-size="12" fill="#3B295D" text-anchor="middle" font-weight="bold">Embedding 층</text><text x="560" y="88" font-size="12" fill="#6B7280" text-anchor="middle" font-weight="bold">학습 가능한 숫자 16개</text><rect x="230" y="96" width="150" height="190" rx="12" ry="12" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="2"/><text x="305" y="178" font-size="15" fill="#3B295D" text-anchor="middle" font-weight="bold">Embedding</text><text x="305" y="198" font-size="10.5" fill="#3B295D" text-anchor="middle" font-weight="normal">학습으로 조정되는</text><text x="305" y="213" font-size="10.5" fill="#3B295D" text-anchor="middle" font-weight="normal">숫자 표</text><rect x="40" y="102" width="60" height="52" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="70" y="138" font-size="22" fill="#0E4A46" text-anchor="middle" font-weight="bold">소</text><line x1="104" y1="128" x2="115" y2="128" stroke="#B6BDC8" stroke-width="1.6"/><polygon points="124,128 114,122 114,134" fill="#B6BDC8"/><rect x="128" y="102" width="76" height="52" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="166" y="136" font-size="19" fill="#1B3A5E" text-anchor="middle" font-weight="bold">27</text><line x1="208" y1="128" x2="219" y2="128" stroke="#B6BDC8" stroke-width="1.6"/><polygon points="228,128 218,122 218,134" fill="#B6BDC8"/><line x1="382" y1="128" x2="393" y2="128" stroke="#B6BDC8" stroke-width="1.6"/><polygon points="402,128 392,122 392,134" fill="#B6BDC8"/><rect x="406" y="102" width="454" height="52" rx="10" ry="10" fill="#F4F1FA" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="420.2" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="448.6" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="476.9" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="505.3" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="533.7" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="562.1" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="590.4" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="618.8" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="647.2" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="675.6" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="703.9" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="732.3" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="760.7" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="789.1" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="817.4" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="845.8" cy="128" r="6" fill="#7B5EA7" opacity="0.8"/><text x="868" y="133" font-size="11" fill="#3B295D" text-anchor="start" font-weight="normal">16개</text><rect x="40" y="164" width="60" height="52" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="70" y="200" font-size="22" fill="#0E4A46" text-anchor="middle" font-weight="bold">득</text><line x1="104" y1="190" x2="115" y2="190" stroke="#B6BDC8" stroke-width="1.6"/><polygon points="124,190 114,184 114,196" fill="#B6BDC8"/><rect x="128" y="164" width="76" height="52" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="166" y="198" font-size="19" fill="#1B3A5E" text-anchor="middle" font-weight="bold">29</text><line x1="208" y1="190" x2="219" y2="190" stroke="#B6BDC8" stroke-width="1.6"/><polygon points="228,190 218,184 218,196" fill="#B6BDC8"/><line x1="382" y1="190" x2="393" y2="190" stroke="#B6BDC8" stroke-width="1.6"/><polygon points="402,190 392,184 392,196" fill="#B6BDC8"/><rect x="406" y="164" width="454" height="52" rx="10" ry="10" fill="#F4F1FA" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="420.2" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="448.6" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="476.9" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="505.3" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="533.7" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="562.1" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="590.4" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="618.8" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="647.2" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="675.6" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="703.9" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="732.3" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="760.7" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="789.1" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="817.4" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="845.8" cy="190" r="6" fill="#7B5EA7" opacity="0.8"/><text x="868" y="195" font-size="11" fill="#3B295D" text-anchor="start" font-weight="normal">16개</text><rect x="40" y="226" width="60" height="52" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="70" y="262" font-size="22" fill="#0E4A46" text-anchor="middle" font-weight="bold">금</text><line x1="104" y1="252" x2="115" y2="252" stroke="#B6BDC8" stroke-width="1.6"/><polygon points="124,252 114,246 114,258" fill="#B6BDC8"/><rect x="128" y="226" width="76" height="52" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="166" y="260" font-size="19" fill="#1B3A5E" text-anchor="middle" font-weight="bold">30</text><line x1="208" y1="252" x2="219" y2="252" stroke="#B6BDC8" stroke-width="1.6"/><polygon points="228,252 218,246 218,258" fill="#B6BDC8"/><line x1="382" y1="252" x2="393" y2="252" stroke="#B6BDC8" stroke-width="1.6"/><polygon points="402,252 392,246 392,258" fill="#B6BDC8"/><rect x="406" y="226" width="454" height="52" rx="10" ry="10" fill="#F4F1FA" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="420.2" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="448.6" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="476.9" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="505.3" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="533.7" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="562.1" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="590.4" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="618.8" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="647.2" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="675.6" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="703.9" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="732.3" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="760.7" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="789.1" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="817.4" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><circle cx="845.8" cy="252" r="6" fill="#7B5EA7" opacity="0.8"/><text x="868" y="257" font-size="11" fill="#3B295D" text-anchor="start" font-weight="normal">16개</text><rect x="40" y="306" width="860" height="118" rx="10" ry="10" fill="#F7F8FA" stroke="#D7DBE2" stroke-width="1.5"/><text x="60" y="334" font-size="13.5" fill="#3C4552" text-anchor="start" font-weight="normal">· 토큰 ID (27, 29, 30) 는 글자를 구분하기 위한 이름표일 뿐입니다.</text><text x="60" y="360" font-size="13.5" fill="#3C4552" text-anchor="start" font-weight="normal">· Embedding 은 그 ID 를 모델이 학습하면서 조정할 수 있는 숫자 16개의 묶음으로 바꿉니다.</text><text x="60" y="386" font-size="13.5" fill="#3C4552" text-anchor="start" font-weight="normal">· 같은 글자에는 항상 같은 16개가, 다른 글자에는 다른 16개가 나옵니다.</text><text x="60" y="412" font-size="13.5" fill="#77400F" text-anchor="start" font-weight="bold">· 이 숫자들은 처음에는 무작위이고, 학습을 거치며 조정됩니다.</text></svg>

### 왜 필요한가요?
토큰 ID 를 그대로 계산에 쓰면 "30번 글자가 27번 글자보다 크다" 는 엉뚱한 의미가 생깁니다.
Embedding 은 **단순한 식별 번호를, 모델이 학습하면서 조정할 수 있는 숫자 묶음으로** 바꿔 줍니다.

### 출력은 무엇인가요?
**글자 토큰 하나마다 숫자 16개**입니다.

### 다음 단계에는 무엇이 넘어가나요?
글자마다 만들어진 이 숫자들이 **GlobalAveragePooling1D** 로 넘어갑니다.

---
## 02-4. 준비 ③ **GlobalAveragePooling1D**

### 지금 무엇이 들어오나요?
글자마다 만들어진 **숫자 16개들**입니다.

### 이 단계에서 무엇을 하나요?
**같은 자리에 있는 숫자끼리 평균**을 냅니다.

<svg viewBox="0 0 940 520" role="img" aria-label="글자별 숫자를 평균 내어 문장 하나의 숫자 16개로 합치는 과정" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:880px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>GlobalAveragePooling1D</title><desc>글자 소, 득, 금, 액 이 각각 가진 숫자 16개를 같은 자리끼리 평균 내어 문장 전체를 나타내는 숫자 16개를 만드는 과정을 보여 주는 그림. 여기서 만들어진 숫자 16개가 MLP 의 실제 입력이라는 설명을 포함한다.</desc><rect x="0" y="0" width="940" height="520" fill="#FFFFFF"/><text x="470" y="32" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">같은 자리끼리 평균 → 문장 하나도 숫자 16개</text><text x="470" y="56" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">글자가 몇 개든 결과는 항상 숫자 16개입니다.</text><line x1="304.4" y1="92" x2="304.4" y2="356" stroke="#C9D3E2" stroke-width="1.4" stroke-dasharray="4 5"/><line x1="473.1" y1="92" x2="473.1" y2="356" stroke="#C9D3E2" stroke-width="1.4" stroke-dasharray="4 5"/><line x1="641.9" y1="92" x2="641.9" y2="356" stroke="#C9D3E2" stroke-width="1.4" stroke-dasharray="4 5"/><text x="120" y="88" font-size="12" fill="#6B7280" text-anchor="middle" font-weight="bold">글자 토큰</text><text x="490" y="88" font-size="12" fill="#6B7280" text-anchor="middle" font-weight="bold">Embedding 이 만든 숫자 16개</text><rect x="64" y="96" width="112" height="40" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="120" y="123" font-size="19" fill="#0E4A46" text-anchor="middle" font-weight="bold">소</text><rect x="220" y="96" width="540" height="40" rx="10" ry="10" fill="#F4F1FA" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="236.9" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="270.6" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="304.4" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="338.1" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="371.9" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="405.6" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="439.4" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="473.1" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="506.9" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="540.6" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="574.4" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="608.1" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="641.9" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="675.6" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="709.4" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="743.1" cy="116" r="6" fill="#7B5EA7" opacity="0.75"/><text x="772" y="121" font-size="11.5" fill="#3B295D" text-anchor="start" font-weight="normal">숫자 16개</text><rect x="64" y="148" width="112" height="40" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="120" y="175" font-size="19" fill="#0E4A46" text-anchor="middle" font-weight="bold">득</text><rect x="220" y="148" width="540" height="40" rx="10" ry="10" fill="#F4F1FA" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="236.9" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="270.6" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="304.4" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="338.1" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="371.9" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="405.6" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="439.4" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="473.1" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="506.9" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="540.6" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="574.4" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="608.1" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="641.9" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="675.6" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="709.4" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="743.1" cy="168" r="6" fill="#7B5EA7" opacity="0.75"/><text x="772" y="173" font-size="11.5" fill="#3B295D" text-anchor="start" font-weight="normal">숫자 16개</text><rect x="64" y="200" width="112" height="40" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="120" y="227" font-size="19" fill="#0E4A46" text-anchor="middle" font-weight="bold">금</text><rect x="220" y="200" width="540" height="40" rx="10" ry="10" fill="#F4F1FA" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="236.9" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="270.6" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="304.4" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="338.1" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="371.9" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="405.6" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="439.4" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="473.1" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="506.9" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="540.6" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="574.4" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="608.1" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="641.9" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="675.6" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="709.4" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="743.1" cy="220" r="6" fill="#7B5EA7" opacity="0.75"/><text x="772" y="225" font-size="11.5" fill="#3B295D" text-anchor="start" font-weight="normal">숫자 16개</text><rect x="64" y="252" width="112" height="40" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="120" y="279" font-size="19" fill="#0E4A46" text-anchor="middle" font-weight="bold">액</text><rect x="220" y="252" width="540" height="40" rx="10" ry="10" fill="#F4F1FA" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="236.9" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="270.6" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="304.4" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="338.1" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="371.9" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="405.6" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="439.4" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="473.1" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="506.9" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="540.6" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="574.4" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="608.1" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="641.9" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="675.6" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="709.4" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><circle cx="743.1" cy="272" r="6" fill="#7B5EA7" opacity="0.75"/><text x="772" y="277" font-size="11.5" fill="#3B295D" text-anchor="start" font-weight="normal">숫자 16개</text><text x="490" y="328" font-size="12.5" fill="#6B7280" text-anchor="middle" font-weight="normal">⋯ 문장의 나머지 글자도 똑같이 숫자 16개를 가지고 있습니다 ⋯</text><line x1="490" y1="338" x2="490" y2="371" stroke="#8A93A0" stroke-width="2"/><polygon points="490,380 484,370 496,370" fill="#8A93A0"/><text x="520" y="356" font-size="12.5" fill="#77400F" text-anchor="start" font-weight="bold">같은 자리에 있는 숫자끼리 평균</text><text x="520" y="373" font-size="11" fill="#77400F" text-anchor="start" font-weight="normal">GlobalAveragePooling1D</text><rect x="64" y="384" width="112" height="48" rx="8" ry="8" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="2"/><text x="120" y="405" font-size="13" fill="#77400F" text-anchor="middle" font-weight="bold">문장 전체</text><text x="120" y="422" font-size="10.5" fill="#77400F" text-anchor="middle" font-weight="normal">1개</text><rect x="220" y="384" width="540" height="48" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="2"/><circle cx="236.9" cy="408" r="7" fill="#4E9A57"/><circle cx="270.6" cy="408" r="7" fill="#4E9A57"/><circle cx="304.4" cy="408" r="7" fill="#4E9A57"/><circle cx="338.1" cy="408" r="7" fill="#4E9A57"/><circle cx="371.9" cy="408" r="7" fill="#4E9A57"/><circle cx="405.6" cy="408" r="7" fill="#4E9A57"/><circle cx="439.4" cy="408" r="7" fill="#4E9A57"/><circle cx="473.1" cy="408" r="7" fill="#4E9A57"/><circle cx="506.9" cy="408" r="7" fill="#4E9A57"/><circle cx="540.6" cy="408" r="7" fill="#4E9A57"/><circle cx="574.4" cy="408" r="7" fill="#4E9A57"/><circle cx="608.1" cy="408" r="7" fill="#4E9A57"/><circle cx="641.9" cy="408" r="7" fill="#4E9A57"/><circle cx="675.6" cy="408" r="7" fill="#4E9A57"/><circle cx="709.4" cy="408" r="7" fill="#4E9A57"/><circle cx="743.1" cy="408" r="7" fill="#4E9A57"/><text x="772" y="413" font-size="11.5" fill="#1D4726" text-anchor="start" font-weight="bold">숫자 16개</text><rect x="64" y="456" width="812" height="44" rx="10" ry="10" fill="#FDF6EE" stroke="#DE8A3E" stroke-width="2"/><text x="470" y="484" font-size="15.5" fill="#77400F" text-anchor="middle" font-weight="bold">★ 여기서 만들어진 숫자 16개가 MLP 의 실제 입력입니다.</text></svg>

### 왜 필요한가요?
문장마다 글자 수가 다르면 MLP 가 받을 입력 크기가 계속 달라집니다.
MLP 는 **입력 개수가 고정**되어야 계산할 수 있습니다.
평균을 내면 글자 수와 상관없이 **항상 숫자 16개**가 됩니다.

### 출력은 무엇인가요?
문장 전체를 나타내는 **숫자 16개**입니다.

---
## 02-5. 여기까지가 준비 — MLP 의 실제 입력

> ### **여기서 만들어진 숫자 16개가 MLP 의 실제 입력입니다.**

| | 어디에 속하나 | 하는 일 |
|---|---|---|
| TextVectorization | 입력 준비 | 문장 → 글자 토큰 → 토큰 ID |
| Embedding | 입력 준비 | 토큰 ID → 글자마다 숫자 16개 |
| GlobalAveragePooling1D | 입력 준비 | 평균 → 문장 하나를 숫자 16개로 |
| **Dense · ReLU · Dense · Softmax** | **★ MLP 핵심** | **숫자 16개 → 3개 후보 점수** |

> ### 매우 중요
> **Embedding 과 Pooling 은 MLP 자체가 아니라, MLP 가 받을 입력을 준비하는 단계입니다.**
> Embedding 안의 값도 학습으로 조정되지만,
> 오늘 배우는 **MLP Dense 층의 Weight/Bias 와는 구분해서** 생각해야 합니다.

여기까지가 준비입니다. 다음 장부터가 **오늘의 핵심**입니다.

---
---
# 03. `★ [MLP 핵심]`  MLP 전체 구조

> **📍 현재 위치**  
> 민원 문장 → 입력 준비 → **★ 여기부터 MLP ← 지금 여기** → Dense → ReLU → Dense → Softmax → 3개 점수

준비가 끝나 **숫자 16개**가 손에 들어왔습니다.
이제 이 숫자 16개가 어떻게 **3개 카테고리 점수**로 바뀌는지 봅니다.

<svg viewBox="0 0 940 620" role="img" aria-label="숫자 16개가 3개 점수가 되는 MLP 내부 흐름" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:820px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>MLP 내부 흐름</title><desc>입력 준비에서 넘어온 숫자 16개가 Dense 16 의 Weight 곱하기 Input 더하기 Bias 와 ReLU 를 지나 은닉층 숫자 16개가 되고, 다시 Dense 3 의 Weight 곱하기 Hidden 더하기 Bias 와 Softmax 를 지나 3개 카테고리 점수가 되는 과정을 보여 주는 흐름도.</desc><rect x="0" y="0" width="940" height="620" fill="#FFFFFF"/><text x="470" y="32" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">숫자 16개는 어떻게 3개 점수가 되나?</text><text x="470" y="56" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">문장을 숫자로 바꾸는 준비 과정은 앞 그림에서 끝났습니다. 여기서부터가 MLP 입니다.</text><rect x="28" y="76" width="884" height="518" rx="14" ry="14" fill="#F2F6FC" stroke="#4C78A8" stroke-width="2.5"/><text x="48" y="102" font-size="14" fill="#2F5C93" text-anchor="start" font-weight="bold">[ ★ MLP 핵심 — 오늘 이해해야 하는 부분 ]</text><rect x="250" y="114" width="440" height="46" rx="8" ry="8" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="2"/><text x="470" y="136" font-size="16" fill="#77400F" text-anchor="middle" font-weight="bold">MLP 입력 : 숫자 16개</text><text x="470" y="153" font-size="11.5" fill="#77400F" text-anchor="middle" font-weight="normal">← 입력 준비(Embedding · Pooling)에서 넘어온 값</text><line x1="470" y1="160" x2="470" y2="175" stroke="#8A93A0" stroke-width="2"/><polygon points="470,184 464,174 476,174" fill="#8A93A0"/><rect x="250" y="184" width="440" height="64" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="2"/><text x="470" y="210" font-size="16.5" fill="#1B3A5E" text-anchor="middle" font-weight="bold">은닉층   Dense(16)</text><text x="470" y="234" font-size="14.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">Weight × Input + Bias</text><text x="706" y="220" font-size="12.5" fill="#4C78A8" text-anchor="start" font-weight="normal">뉴런 16개</text><line x1="470" y1="248" x2="470" y2="263" stroke="#8A93A0" stroke-width="2"/><polygon points="470,272 464,262 476,262" fill="#8A93A0"/><rect x="250" y="272" width="440" height="44" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="2"/><text x="470" y="300" font-size="15" fill="#0E4A46" text-anchor="middle" font-weight="bold">ReLU   (음수는 0, 양수는 그대로)</text><line x1="470" y1="316" x2="470" y2="331" stroke="#8A93A0" stroke-width="2"/><polygon points="470,340 464,330 476,330" fill="#8A93A0"/><rect x="250" y="340" width="440" height="42" rx="8" ry="8" fill="#FFFFFF" stroke="#4C78A8" stroke-width="1.5"/><text x="470" y="367" font-size="15" fill="#1B3A5E" text-anchor="middle" font-weight="bold">은닉층 출력 : 숫자 16개</text><line x1="470" y1="382" x2="470" y2="397" stroke="#8A93A0" stroke-width="2"/><polygon points="470,406 464,396 476,396" fill="#8A93A0"/><rect x="250" y="406" width="440" height="64" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="2"/><text x="470" y="432" font-size="16.5" fill="#1D4726" text-anchor="middle" font-weight="bold">출력층   Dense(3)</text><text x="470" y="456" font-size="14.5" fill="#1D4726" text-anchor="middle" font-weight="normal">Weight × Hidden + Bias</text><text x="706" y="442" font-size="12.5" fill="#4E9A57" text-anchor="start" font-weight="normal">뉴런 3개</text><line x1="470" y1="470" x2="470" y2="485" stroke="#8A93A0" stroke-width="2"/><polygon points="470,494 464,484 476,484" fill="#8A93A0"/><rect x="250" y="494" width="440" height="44" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="2"/><text x="470" y="522" font-size="15" fill="#0E4A46" text-anchor="middle" font-weight="bold">Softmax   (합이 1인 비교용 점수로)</text><line x1="470" y1="538" x2="470" y2="553" stroke="#8A93A0" stroke-width="2"/><polygon points="470,562 464,552 476,552" fill="#8A93A0"/><rect x="250" y="562" width="440" height="44" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="2"/><text x="470" y="590" font-size="16" fill="#1D4726" text-anchor="middle" font-weight="bold">3개 카테고리 점수</text></svg>

### 그림에서 확인할 점
- MLP 의 시작은 문장이 아니라 **숫자 16개**입니다.
- MLP 안에서 일어나는 계산은 딱 두 가지의 반복입니다.
  **① Weight × Input + Bias**, **② 활성화 함수(ReLU / Softmax)**
- 층을 두 번 지나면 숫자 16개가 **3개 점수**가 됩니다.

---

### 이제 코드를 봅니다

아래 모델 정의에서 **오늘 이해해야 하는 것은 마지막 두 줄(Dense 두 개)뿐**입니다.

In [4]:
model = tf.keras.Sequential([

    # ============================================================
    # [입력 준비 — MLP 자체는 아닙니다]
    # ============================================================

    # 민원 문장을 글자 그대로 입력받습니다.
    tf.keras.Input(shape=(1,), dtype=tf.string),

    # 준비 ① 문장 → 글자 토큰 → 토큰 ID
    vectorize,

    # 준비 ② 토큰 ID → 학습 가능한 숫자 표현 (글자 하나마다 숫자 16개)
    #   mask_zero=True : 문장이 짧아 0(빈칸)으로 채워진 자리는
    #                    실제 글자처럼 취급하지 않고 평균에서 제외합니다.
    tf.keras.layers.Embedding(2000, 16, mask_zero=True),

    # 준비 ③ 글자별 숫자를 평균 내어 문장 하나를 숫자 16개로
    tf.keras.layers.GlobalAveragePooling1D(),

    # ============================================================
    # ★ MLP 핵심 시작
    # ============================================================

    # 은닉층
    # 입력 숫자 16개
    # → Weight × Input + Bias
    # → ReLU
    # → 새로운 숫자 16개
    tf.keras.layers.Dense(16, activation="relu"),

    # 출력층
    # 은닉층 숫자 16개
    # → Weight × Hidden + Bias
    # → Softmax
    # → 3개 카테고리 점수
    tf.keras.layers.Dense(3, activation="softmax"),
])

print("MLP 모델이 만들어졌습니다.")

MLP 모델이 만들어졌습니다.


---
---
# 04. `★ [MLP 핵심]`  Dense 안에서는 무엇이 일어날까?

> **📍 현재 위치**  
> 민원 문장 → 입력 준비 → **★ Dense 내부 ← 지금 여기** → ReLU → Dense(3) → Softmax → 3개 점수

## 04-1. 뉴런 하나는 무엇을 계산할까?

`Dense(16, activation="relu")` 는 코드로는 **한 줄**이지만,
그 안에서는 **뉴런 하나하나가 아래 계산**을 하고 있습니다.

> ### 주의
> 아래 숫자는 **설명을 위해 만든 작은 예제**입니다.
> 실제 모델은 숫자 16개를 다루며, **아래 값은 실제 모델의 출력값이 아닙니다.**

<svg viewBox="0 0 940 470" role="img" aria-label="뉴런 하나가 하는 계산" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:880px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>뉴런 하나가 하는 계산</title><desc>설명용 작은 숫자 예제. 입력 0.8, 0.3, 0.5에 각각 Weight 0.4, -0.2, 0.6을 곱해 더하고 Bias 0.1을 더하면 0.66이 되며 ReLU를 통과해도 0.66이 그대로 남는 과정을 보여 주는 그림.</desc><rect x="0" y="0" width="940" height="470" fill="#FFFFFF"/><text x="470" y="32" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">뉴런 하나는 이렇게 계산합니다</text><text x="470" y="56" font-size="13" fill="#B04A2E" text-anchor="middle" font-weight="normal">설명용으로 만든 작은 숫자 예제입니다 — 실제 모델의 출력값이 아닙니다</text><rect x="60" y="104" width="130" height="52" rx="8" ry="8" fill="#EDEFF3" stroke="#AEB6C2" stroke-width="1.5"/><text x="125" y="137" font-size="18" fill="#3D4552" text-anchor="middle" font-weight="bold">x₁ = 0.8</text><line x1="190" y1="130" x2="290" y2="130" stroke="#8A93A0" stroke-width="2"/><polygon points="299,130 289,124 289,136" fill="#8A93A0"/><text x="240" y="120" font-size="13" fill="#7B5EA7" text-anchor="middle" font-weight="bold">× w₁=0.4</text><rect x="60" y="172" width="130" height="52" rx="8" ry="8" fill="#EDEFF3" stroke="#AEB6C2" stroke-width="1.5"/><text x="125" y="205" font-size="18" fill="#3D4552" text-anchor="middle" font-weight="bold">x₂ = 0.3</text><line x1="190" y1="198" x2="290" y2="198" stroke="#8A93A0" stroke-width="2"/><polygon points="299,198 289,192 289,204" fill="#8A93A0"/><text x="240" y="188" font-size="13" fill="#7B5EA7" text-anchor="middle" font-weight="bold">× w₂=-0.2</text><rect x="60" y="240" width="130" height="52" rx="8" ry="8" fill="#EDEFF3" stroke="#AEB6C2" stroke-width="1.5"/><text x="125" y="273" font-size="18" fill="#3D4552" text-anchor="middle" font-weight="bold">x₃ = 0.5</text><line x1="190" y1="266" x2="290" y2="266" stroke="#8A93A0" stroke-width="2"/><polygon points="299,266 289,260 289,272" fill="#8A93A0"/><text x="240" y="256" font-size="13" fill="#7B5EA7" text-anchor="middle" font-weight="bold">× w₃=0.6</text><rect x="300" y="96" width="250" height="196" rx="12" ry="12" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="2"/><text x="425" y="124" font-size="15" fill="#3B295D" text-anchor="middle" font-weight="bold">곱해서 전부 더하기</text><text x="425" y="152" font-size="14" fill="#3B295D" text-anchor="middle" font-weight="normal" font-family="Consolas,monospace">0.8 × 0.4    =  0.32</text><text x="425" y="175" font-size="14" fill="#3B295D" text-anchor="middle" font-weight="normal" font-family="Consolas,monospace">0.3 × (-0.2) = -0.06</text><text x="425" y="198" font-size="14" fill="#3B295D" text-anchor="middle" font-weight="normal" font-family="Consolas,monospace">0.5 × 0.6    =  0.30</text><line x1="330" y1="215" x2="520" y2="215" stroke="#7B5EA7" stroke-width="1.2" opacity="0.6"/><text x="425" y="244" font-size="14" fill="#3B295D" text-anchor="middle" font-weight="normal" font-family="Consolas,monospace">합계          =  0.56</text><text x="425" y="267" font-size="14" fill="#3B295D" text-anchor="middle" font-weight="normal" font-family="Consolas,monospace">Bias  b      = +0.10</text><line x1="550" y1="200" x2="620" y2="200" stroke="#8A93A0" stroke-width="2"/><polygon points="629,200 619,194 619,206" fill="#8A93A0"/><rect x="630" y="172" width="120" height="56" rx="10" ry="10" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="2"/><text x="690" y="207" font-size="22" fill="#77400F" text-anchor="middle" font-weight="bold">0.66</text><text x="690" y="160" font-size="12.5" fill="#77400F" text-anchor="middle" font-weight="normal">Weight·Bias 계산 결과</text><line x1="690" y1="228" x2="690" y2="264" stroke="#8A93A0" stroke-width="2"/><polygon points="690,273 684,263 696,263" fill="#8A93A0"/><rect x="600" y="273" width="180" height="50" rx="10" ry="10" fill="#E1F3F2" stroke="#2E9E96" stroke-width="2"/><text x="690" y="304" font-size="18" fill="#0E4A46" text-anchor="middle" font-weight="bold">ReLU</text><line x1="690" y1="323" x2="690" y2="359" stroke="#8A93A0" stroke-width="2"/><polygon points="690,368 684,358 696,358" fill="#8A93A0"/><rect x="630" y="368" width="120" height="52" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="2"/><text x="690" y="401" font-size="22" fill="#1D4726" text-anchor="middle" font-weight="bold">0.66</text><text x="690" y="440" font-size="13.5" fill="#1D4726" text-anchor="middle" font-weight="bold">이 뉴런의 출력</text><rect x="60" y="330" width="500" height="100" rx="10" ry="10" fill="#F7F8FA" stroke="#D7DBE2" stroke-width="1.5"/><text x="80" y="356" font-size="14" fill="#3D4552" text-anchor="start" font-weight="bold">ReLU 는 이렇게 동작합니다</text><text x="80" y="382" font-size="13.5" fill="#5A6270" text-anchor="start" font-weight="normal">계산 결과가 음수이면  →  0 으로 바꿉니다</text><text x="80" y="405" font-size="13.5" fill="#5A6270" text-anchor="start" font-weight="normal">계산 결과가 양수이면  →  그대로 통과시킵니다</text><text x="80" y="424" font-size="12.5" fill="#8B93A0" text-anchor="start" font-weight="normal">여기서는 0.66 이 양수이므로 0.66 이 그대로 남았습니다.</text></svg>

### 계산을 표로 옮기면

| 입력 | Weight | 곱한 값 |
|---:|---:|---:|
| 0.8 | 0.4 | 0.32 |
| 0.3 | -0.2 | -0.06 |
| 0.5 | 0.6 | 0.30 |
| | **합계** | **0.56** |
| | **+ Bias** | **+0.10** |
| | **계산 결과** | **0.66** |
| | **ReLU 통과 후** | **0.66** ← 이 뉴런의 출력 |

> **Dense 안의 뉴런 하나는
> 입력에 Weight 를 곱하고 Bias 를 더한 뒤, 활성화 함수를 통과시킵니다.**

---
### 04-1A. Weight (가중치) 는 무엇인가요?

> **각 입력값이 이 뉴런의 계산 결과에 얼마나, 그리고 어느 방향으로 영향을 줄지를 조정하는 값입니다.**
> (쉽게 말하면 **입력마다 서로 다른 영향력**을 주는 값입니다.)

- 입력 **하나마다 하나씩** 있습니다. 입력이 16개면 Weight 도 16개입니다.
- **양수 Weight** 는 그 입력이 커질수록 결과를 **올리는** 방향으로,
  **음수 Weight** 는 결과를 **내리는** 방향으로 작용합니다.
- 앞 예제에서 `w₂ = -0.2` 였기 때문에 두 번째 입력은 결과를 **깎는** 쪽으로 작용했습니다.

> Weight 는 처음에는 **무작위**입니다. **학습을 통해 조정되는 대표적인 값**입니다.

---
### 04-1B. Bias (편향) 는 무엇인가요?

> **여러 (입력 × Weight) 를 모두 더한 결과 전체를 조금 위나 아래로 이동시키는 값입니다.**

- **뉴런마다 하나씩** 있습니다. 입력이 몇 개든 Bias 는 뉴런당 1개입니다.
- 앞 예제에서 합계 `0.56` 에 Bias `+0.10` 을 더해 `0.66` 이 되었습니다.
- 보조 설명으로는 **"이 뉴런이 반응하기 쉬운 정도를 조정하는 값"** 정도로 이해해도 좋습니다.

| | Weight | Bias |
|---|---|---|
| 몇 개인가 | 입력 **하나마다** 하나 | 뉴런 **하나마다** 하나 |
| 하는 일 | 각 입력의 영향력과 방향 조정 | 결과 전체를 위/아래로 이동 |
| 학습으로 바뀌나 | **네** | **네** |

---
### 04-1C. ReLU 는 무엇을 하나요?

> **음수는 0 으로 만들고, 양수는 그대로 전달하는 함수입니다.**

ReLU 는 **은닉층 계산이 끝난 직후, 다음 층으로 넘기기 직전**에 놓입니다.
바로 앞 그림(뉴런 하나가 하는 계산)의 **아래쪽 절반**이 이 단계입니다.

| Weight·Bias 계산 결과 | ReLU 통과 후 |
|---:|---:|
| -1.20 | **0** |
| -0.05 | **0** |
| 0.00 | 0 |
| 0.66 | **0.66** |
| 2.30 | **2.30** |

**입력** : Weight 와 Bias 계산 결과(음수일 수도, 양수일 수도 있음)
**출력** : 0 이상의 값 → 그대로 **다음 층으로 전달**

> **[선택 학습]** ReLU 를 "필요한 정보만 남긴다" 라고 단정하기는 어렵습니다.
> 더 정확하게는, **단순한 비선형 변환을 끼워 넣어 여러 층을 쌓았을 때 복잡한 패턴을 학습할 수 있게 만드는** 장치입니다.
> 활성화 함수가 없으면 층을 아무리 쌓아도 결국 하나의 선형 변환과 같아집니다.

---
## 04-2. 뉴런 16개가 모이면 Dense(16)

### 지금 무엇이 들어오나요?
입력 준비가 만들어 준 **숫자 16개**입니다.

### 이 단계에서 무엇을 하나요?
방금 본 계산을 하는 **뉴런이 16개** 있고, 각 뉴런이 **자기만의 Weight 와 Bias** 로 계산합니다.

<svg viewBox="0 0 940 552" role="img" aria-label="뉴런 16개가 모여 Dense 16 층이 되는 구조" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:860px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>Dense(16) 전체 구조</title><desc>입력 숫자 16개를 뉴런 16개가 모두 함께 받고, 각 뉴런이 자기만의 Weight 와 Bias 로 서로 다른 계산을 한 뒤 ReLU 를 거쳐 숫자 하나씩을 내놓아 출력 숫자 16개가 되는 구조를 보여 주는 그림.</desc><rect x="0" y="0" width="940" height="552" fill="#FFFFFF"/><text x="470" y="32" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">뉴런 16개가 모이면 Dense(16)</text><text x="470" y="56" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">입력 16개를 그대로 복사하는 것이 아니라, 서로 다른 16개의 새 숫자를 만듭니다.</text><rect x="40" y="78" width="140" height="44" rx="8" ry="8" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="2"/><text x="110" y="100" font-size="12.5" fill="#77400F" text-anchor="middle" font-weight="bold">MLP 입력</text><text x="110" y="115" font-size="10.5" fill="#77400F" text-anchor="middle" font-weight="normal">숫자 16개</text><rect x="200" y="78" width="560" height="44" rx="10" ry="10" fill="#F4F1FA" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="217.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="252.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="287.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="322.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="357.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="392.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="427.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="462.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="497.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="532.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="567.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="602.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="637.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="672.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="707.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><circle cx="742.5" cy="100" r="7" fill="#7B5EA7" opacity="0.8"/><text x="772" y="105" font-size="11.5" fill="#3B295D" text-anchor="start" font-weight="normal">16개</text><line x1="470" y1="122" x2="470" y2="139" stroke="#8A93A0" stroke-width="2"/><polygon points="470,148 464,138 476,138" fill="#8A93A0"/><text x="494" y="140" font-size="11.5" fill="#6B7280" text-anchor="start" font-weight="normal">뉴런 16개가 이 16개를 모두 받습니다</text><rect x="200" y="148" width="560" height="246" rx="12" ry="12" fill="#F2F6FC" stroke="#4C78A8" stroke-width="2.5"/><text x="220" y="174" font-size="14.5" fill="#2F5C93" text-anchor="start" font-weight="bold">Dense(16)  —  뉴런 16개</text><rect x="220" y="186" width="520" height="34" rx="6" ry="6" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.2"/><text x="268" y="208" font-size="13" fill="#1B3A5E" text-anchor="middle" font-weight="bold">뉴런 1</text><text x="480" y="208" font-size="12.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">자기 Weight · Bias 로 계산  →  ReLU  →  숫자 1개</text><rect x="220" y="226" width="520" height="34" rx="6" ry="6" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.2"/><text x="268" y="248" font-size="13" fill="#1B3A5E" text-anchor="middle" font-weight="bold">뉴런 2</text><text x="480" y="248" font-size="12.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">자기 Weight · Bias 로 계산  →  ReLU  →  숫자 1개</text><rect x="220" y="306" width="520" height="34" rx="6" ry="6" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.2"/><text x="268" y="328" font-size="13" fill="#1B3A5E" text-anchor="middle" font-weight="bold">뉴런 16</text><text x="480" y="328" font-size="12.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">자기 Weight · Bias 로 계산  →  ReLU  →  숫자 1개</text><text x="480" y="288" font-size="18" fill="#4C78A8" text-anchor="middle" font-weight="bold">⋮</text><text x="480" y="366" font-size="12" fill="#2F5C93" text-anchor="middle" font-weight="normal">뉴런마다 Weight 와 Bias 가 다르기 때문에 결과도 서로 다릅니다</text><line x1="470" y1="394" x2="470" y2="411" stroke="#8A93A0" stroke-width="2"/><polygon points="470,420 464,410 476,410" fill="#8A93A0"/><rect x="40" y="420" width="140" height="44" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="2"/><text x="110" y="442" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="bold">은닉층 출력</text><text x="110" y="457" font-size="10.5" fill="#1D4726" text-anchor="middle" font-weight="normal">숫자 16개</text><rect x="200" y="420" width="560" height="44" rx="10" ry="10" fill="#EFF7F0" stroke="#4E9A57" stroke-width="1.5"/><circle cx="217.5" cy="442" r="7" fill="#4E9A57"/><circle cx="252.5" cy="442" r="7" fill="#4E9A57"/><circle cx="287.5" cy="442" r="7" fill="#4E9A57"/><circle cx="322.5" cy="442" r="7" fill="#4E9A57"/><circle cx="357.5" cy="442" r="7" fill="#4E9A57"/><circle cx="392.5" cy="442" r="7" fill="#4E9A57"/><circle cx="427.5" cy="442" r="7" fill="#4E9A57"/><circle cx="462.5" cy="442" r="7" fill="#4E9A57"/><circle cx="497.5" cy="442" r="7" fill="#4E9A57"/><circle cx="532.5" cy="442" r="7" fill="#4E9A57"/><circle cx="567.5" cy="442" r="7" fill="#4E9A57"/><circle cx="602.5" cy="442" r="7" fill="#4E9A57"/><circle cx="637.5" cy="442" r="7" fill="#4E9A57"/><circle cx="672.5" cy="442" r="7" fill="#4E9A57"/><circle cx="707.5" cy="442" r="7" fill="#4E9A57"/><circle cx="742.5" cy="442" r="7" fill="#4E9A57"/><text x="772" y="447" font-size="11.5" fill="#1D4726" text-anchor="start" font-weight="normal">16개</text><text x="470" y="496" font-size="14" fill="#3C4552" text-anchor="middle" font-weight="normal">뉴런 하나가 숫자 하나를 만들고, 뉴런이 16개이므로 출력도 숫자 16개입니다.</text><text x="470" y="522" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">이 16개가 다음 층인 출력층 Dense(3) 으로 넘어갑니다.</text></svg>

### `Dense(16)` 은 입력 16개를 그대로 복사한다는 뜻이 아닙니다

- 16개 뉴런은 **같은 입력 숫자 16개를 모두** 받습니다.
- 하지만 뉴런마다 **Weight 와 Bias 가 다르므로 결과도 서로 다릅니다.**
- 그래서 **같은 입력에서 서로 다른 16개의 새로운 숫자**가 만들어집니다.

이것을 흔히 "뉴런마다 서로 다른 관점을 만든다" 라고 표현합니다.
정확히 말하면, **각 뉴런이 서로 다른 Weight 와 Bias 를 학습하기 때문에 같은 입력을 서로 다른 방식으로 변환하는 것**입니다.

### 출력은 무엇인가요?
뉴런 16개가 각각 숫자 하나씩을 내놓으므로 **숫자 16개**입니다.

### 다음 단계에는 무엇이 넘어가나요?
이 **은닉층 숫자 16개**가 출력층 `Dense(3)` 으로 넘어갑니다.

---
## 04-3. 출력층 Dense(3)

### 지금 무엇이 들어오나요?
은닉층이 만든 **숫자 16개**입니다.

### 이 단계에서 무엇을 하나요?
출력층도 은닉층과 **똑같은 계산(Weight × 입력 + Bias)** 을 합니다.
다만 뉴런이 **카테고리 수만큼 3개**입니다.

### 출력은 무엇인가요?
카테고리마다 하나씩, **3개의 "원래 점수"** 입니다.
아직 서로 비교하기 좋은 형태는 아닙니다.

<svg viewBox="0 0 940 592" role="img" aria-label="은닉층 숫자 16개가 3개 후보 점수가 되는 과정" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:860px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>Dense(3) 과 Softmax</title><desc>은닉층 숫자 16개를 출력 뉴런 3개가 받아 카테고리별 원래 점수 2.8, 0.4, -0.2 를 만들고, Softmax 가 이를 합이 1 인 비교하기 쉬운 점수 0.86, 0.10, 0.04 로 바꾸는 과정을 보여 주는 그림. 설명용 예시 숫자이며 실제 모델 출력이 아니라는 안내를 포함한다.</desc><rect x="0" y="0" width="940" height="592" fill="#FFFFFF"/><text x="470" y="32" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">숫자 16개는 어떻게 3개 후보 점수가 되나?</text><text x="470" y="56" font-size="13" fill="#B04A2E" text-anchor="middle" font-weight="normal">아래 숫자는 설명용 예시입니다 — 실제 모델의 출력값이 아닙니다</text><rect x="40" y="78" width="150" height="44" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="115" y="100" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="bold">은닉층 출력</text><text x="115" y="115" font-size="10.5" fill="#1D4726" text-anchor="middle" font-weight="normal">숫자 16개</text><rect x="210" y="78" width="550" height="44" rx="10" ry="10" fill="#EFF7F0" stroke="#4E9A57" stroke-width="1.5"/><circle cx="227.2" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="261.6" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="295.9" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="330.3" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="364.7" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="399.1" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="433.4" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="467.8" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="502.2" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="536.6" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="570.9" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="605.3" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="639.7" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="674.1" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="708.4" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><circle cx="742.8" cy="100" r="7" fill="#4E9A57" opacity="0.8"/><line x1="470" y1="122" x2="470" y2="141" stroke="#8A93A0" stroke-width="2"/><polygon points="470,150 464,140 476,140" fill="#8A93A0"/><rect x="150" y="150" width="640" height="172" rx="12" ry="12" fill="#F2F6FC" stroke="#4C78A8" stroke-width="2.5"/><text x="170" y="176" font-size="14.5" fill="#2F5C93" text-anchor="start" font-weight="bold">출력층 Dense(3)  —  뉴런 3개 (카테고리 수만큼)</text><text x="300" y="202" font-size="11.5" fill="#4C78A8" text-anchor="middle" font-weight="normal">각 뉴런이 자기 Weight · Bias 로 계산</text><text x="630" y="202" font-size="11.5" fill="#4C78A8" text-anchor="middle" font-weight="normal">카테고리별 원래 점수</text><rect x="170" y="212" width="180" height="28" rx="6" ry="6" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.2"/><text x="260" y="231" font-size="12.5" fill="#1B3A5E" text-anchor="middle" font-weight="bold">출력 뉴런 1</text><line x1="354" y1="226" x2="391" y2="226" stroke="#B6BDC8" stroke-width="1.6"/><polygon points="400,226 390,220 390,232" fill="#B6BDC8"/><text x="410" y="231" font-size="13" fill="#1B3A5E" text-anchor="start" font-weight="normal">증명서 발급</text><rect x="560" y="212" width="90" height="28" rx="6" ry="6" fill="#FFFFFF" stroke="#4C78A8" stroke-width="1.2"/><text x="605" y="231" font-size="14" fill="#1B3A5E" text-anchor="middle" font-weight="bold">2.8</text><rect x="170" y="246" width="180" height="28" rx="6" ry="6" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.2"/><text x="260" y="265" font-size="12.5" fill="#1B3A5E" text-anchor="middle" font-weight="bold">출력 뉴런 2</text><line x1="354" y1="260" x2="391" y2="260" stroke="#B6BDC8" stroke-width="1.6"/><polygon points="400,260 390,254 390,266" fill="#B6BDC8"/><text x="410" y="265" font-size="13" fill="#1B3A5E" text-anchor="start" font-weight="normal">신고·납부</text><rect x="560" y="246" width="90" height="28" rx="6" ry="6" fill="#FFFFFF" stroke="#4C78A8" stroke-width="1.2"/><text x="605" y="265" font-size="14" fill="#1B3A5E" text-anchor="middle" font-weight="bold">0.4</text><rect x="170" y="280" width="180" height="28" rx="6" ry="6" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.2"/><text x="260" y="299" font-size="12.5" fill="#1B3A5E" text-anchor="middle" font-weight="bold">출력 뉴런 3</text><line x1="354" y1="294" x2="391" y2="294" stroke="#B6BDC8" stroke-width="1.6"/><polygon points="400,294 390,288 390,300" fill="#B6BDC8"/><text x="410" y="299" font-size="13" fill="#1B3A5E" text-anchor="start" font-weight="normal">홈택스 이용</text><rect x="560" y="280" width="90" height="28" rx="6" ry="6" fill="#FFFFFF" stroke="#4C78A8" stroke-width="1.2"/><text x="605" y="299" font-size="14" fill="#1B3A5E" text-anchor="middle" font-weight="bold">-0.2</text><text x="664" y="248" font-size="11" fill="#6B7280" text-anchor="start" font-weight="normal">원래 점수는 크기가</text><text x="664" y="265" font-size="11" fill="#6B7280" text-anchor="start" font-weight="normal">제각각이라 서로</text><text x="664" y="282" font-size="11" fill="#6B7280" text-anchor="start" font-weight="normal">비교하기 어렵습니다</text><line x1="470" y1="322" x2="470" y2="341" stroke="#8A93A0" stroke-width="2"/><polygon points="470,350 464,340 476,340" fill="#8A93A0"/><rect x="310" y="350" width="320" height="48" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="2"/><text x="470" y="372" font-size="16" fill="#0E4A46" text-anchor="middle" font-weight="bold">Softmax</text><text x="470" y="390" font-size="11" fill="#0E4A46" text-anchor="middle" font-weight="normal">세 값을 비교하기 쉬운 형태로</text><line x1="470" y1="398" x2="470" y2="417" stroke="#8A93A0" stroke-width="2"/><polygon points="470,426 464,416 476,416" fill="#8A93A0"/><rect x="150" y="426" width="640" height="130" rx="12" ry="12" fill="#F7F8FA" stroke="#D7DBE2" stroke-width="1.5"/><text x="170" y="462" font-size="13.5" fill="#1D4726" text-anchor="start" font-weight="bold">증명서 발급</text><rect x="300" y="448" width="300" height="18" rx="4" fill="#E4E8EE"/><rect x="300" y="448" width="258.0" height="18" rx="4" fill="#4E9A57"/><text x="614" y="462" font-size="14" fill="#1D4726" text-anchor="start" font-weight="bold">0.86</text><text x="170" y="496" font-size="13.5" fill="#5A6270" text-anchor="start" font-weight="normal">신고·납부</text><rect x="300" y="482" width="300" height="18" rx="4" fill="#E4E8EE"/><rect x="300" y="482" width="30.0" height="18" rx="4" fill="#B9C0CA"/><text x="614" y="496" font-size="14" fill="#5A6270" text-anchor="start" font-weight="normal">0.10</text><text x="170" y="530" font-size="13.5" fill="#5A6270" text-anchor="start" font-weight="normal">홈택스 이용</text><rect x="300" y="516" width="300" height="18" rx="4" fill="#E4E8EE"/><rect x="300" y="516" width="12.0" height="18" rx="4" fill="#B9C0CA"/><text x="614" y="530" font-size="14" fill="#5A6270" text-anchor="start" font-weight="normal">0.04</text><text x="690" y="496" font-size="14" fill="#77400F" text-anchor="start" font-weight="bold">합계 = 1</text><text x="470" y="578" font-size="13.5" fill="#3C4552" text-anchor="middle" font-weight="normal">가장 높은 점수의 카테고리를 후보로 제안합니다. 최종 확인은 담당자가 합니다.</text></svg>

> ### `Dense(3)` 은 왜 3개인가요?
> 이번 실습에서 분류할 카테고리가 **증명서 발급 · 신고·납부 · 홈택스 이용 3개**이기 때문입니다.
> 카테고리가 5개라면 `Dense(5)` 가 됩니다.

---
## 04-4. Softmax

> **여러 후보 점수를 서로 비교하기 쉬운 형태로 바꾸는 함수입니다.**

Softmax 는 **Dense 계산 그 자체가 아닙니다.** 두 단계는 분리해서 이해해야 합니다.

| 단계 | 하는 일 | 결과 |
|---|---|---|
| **Dense(3)** | Weight × 입력 + Bias 를 3번 계산 | 원래 점수 3개 (크기 제각각) |
| **Softmax** | 세 값을 비교하기 쉬운 형태로 변환 | **합이 1 인 점수 3개** |

### 앞 그림의 숫자로 보면

| 카테고리 | Dense(3) 원래 점수 | → Softmax 후 |
|---|---:|---:|
| 증명서 발급 | 2.8 | **0.86** |
| 신고·납부 | 0.4 | 0.10 |
| 홈택스 이용 | -0.2 | 0.04 |
| | | **합계 = 1** |

**가장 높은 점수를 가진 카테고리를 모델이 후보로 제안합니다.**

> ### 주의
> Softmax 로 나온 값을 **"신뢰도"** 나 **"실제 확률"** 이라고 단정하면 안 됩니다.
> "0.86 이니까 86% 확률로 정답" 이라고 읽지 마세요.
> 세 후보를 **서로 비교하기 쉽게 바꾼 점수**로 이해하는 것이 안전합니다.

여기까지가 **Forward(계산)** 입니다.
그런데 지금 모델의 학습 가능한 값은 아직 **무작위**입니다. 결과를 직접 확인해 봅니다.

---
---
# 05. `[결과 확인]`  학습 전에는 어떻게 예측할까?

**아직 아무것도 학습하지 않은** 모델에 민원 문장을 하나 넣어 봅니다.

지금 모델 안의 학습 가능한 값은 **전부 무작위**입니다.

> **학습 전에는 아직 유용한 패턴을 배우지 못했기 때문에,
> 이번 실행에서는 세 카테고리 점수가 서로 비슷하게 나타납니다.**

카테고리가 3개이고 Softmax 는 합을 1 로 만들기 때문에,
구분할 근거가 없으면 셋으로 고르게 나뉜 값 근처에 머무르는 경우가 많습니다.
(다만 무작위 초기화라고 해서 **항상 정확히 1/3 씩** 나오는 것은 아닙니다.)

이 결과는 **학습이 끝난 뒤 같은 문장으로 다시 비교**하므로 기억해 두세요.

In [5]:
# ============================================================
# [결과 확인] 학습하기 전의 모델에 문장 하나를 넣어 봅니다.
#
# 입력 : 민원 문장 1개
# 처리 : 준비(토큰 ID → Embedding → Pooling) → MLP(Dense → ReLU → Dense → Softmax)
# 출력 : 3개 카테고리 점수
#
# 학습이 끝난 뒤 똑같은 문장을 다시 넣어 비교할 것입니다.
# ============================================================
비교문장 = "소득금액증명서를 출력하고 싶습니다."

# np.array(..., dtype=object) : 문장 목록을 모델에 넣을 수 있는 형태로 감싸 줍니다.
# [0] : 문장 하나만 넣었으므로 첫 번째 결과를 꺼냅니다.
학습전점수 = model.predict(np.array([비교문장], dtype=object), verbose=0)[0]

# ------------------------------------------------------------
# [결과 보기용 보조 코드 — MLP 핵심 코드가 아닙니다]
# 아래는 결과를 보기 쉽게 출력하는 부분입니다.
# ------------------------------------------------------------
print(비교문장)
print(CLASS_NAMES)
print(np.round(학습전점수, 3), "  ← 세 점수가 서로 비슷합니다")

소득금액증명서를 출력하고 싶습니다.
['증명서 발급', '신고·납부', '홈택스 이용']
[0.33  0.334 0.336]   ← 세 점수가 서로 비슷합니다


---
---
# 06. `★ [딥러닝 학습 핵심]`  딥러닝은 어떻게 학습할까?

> **📍 현재 위치**  
> Forward → 예측 → Loss → **★ 역전파 · Optimizer ← 지금 여기** → 학습 파라미터 수정 → ↺ 다시 Forward

## 06-1. 학습 한 바퀴

학습 전 모델은 세 카테고리를 거의 구분하지 못했습니다.
그렇다면 **무엇을 어떻게 바꿔야** 맞히게 될까요?

<svg viewBox="0 0 940 560" role="img" aria-label="신경망 학습 반복 과정" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>신경망 학습 반복 과정</title><desc>예측, 정답과 비교, Loss 계산, 역전파, Optimizer의 Weight 수정, 다시 예측 순으로 순환하는 학습 과정을 나타낸 그림. 역전파는 영향을 계산하는 단계이고 Optimizer는 값을 실제로 수정하는 단계로 구분되어 있다.</desc><rect x="0" y="0" width="940" height="560" fill="#FFFFFF"/><text x="470.0" y="32" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">모델은 이렇게 반복해서 학습합니다</text><text x="470.0" y="56" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">model.fit() 안에서 아래 한 바퀴가 계속 돌아갑니다</text><rect x="25" y="84" width="250" height="128" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="51" cy="110" r="13" fill="#7B5EA7"/><text x="51" y="115" font-size="13" fill="#FFFFFF" text-anchor="middle" font-weight="bold">1</text><text x="164.0" y="115" font-size="16.5" fill="#3B295D" text-anchor="middle" font-weight="bold">예측</text><text x="150.0" y="135" font-size="12.5" fill="#3B295D" text-anchor="middle" font-weight="normal" opacity="0.8">model 출력</text><text x="150.0" y="158" font-size="13" fill="#3B295D" text-anchor="middle" font-weight="normal">지금의 Weight와 Bias로</text><text x="150.0" y="177" font-size="13" fill="#3B295D" text-anchor="middle" font-weight="normal">카테고리 점수를 계산</text><rect x="345" y="84" width="250" height="128" rx="10" ry="10" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><circle cx="371" cy="110" r="13" fill="#4C78A8"/><text x="371" y="115" font-size="13" fill="#FFFFFF" text-anchor="middle" font-weight="bold">2</text><text x="484.0" y="115" font-size="16.5" fill="#1B3A5E" text-anchor="middle" font-weight="bold">정답과 비교</text><text x="470.0" y="135" font-size="12.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal" opacity="0.8">y_train</text><text x="470.0" y="158" font-size="13" fill="#1B3A5E" text-anchor="middle" font-weight="normal">실제 카테고리와</text><text x="470.0" y="177" font-size="13" fill="#1B3A5E" text-anchor="middle" font-weight="normal">예측 결과를 대조</text><rect x="665" y="84" width="250" height="128" rx="10" ry="10" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><circle cx="691" cy="110" r="13" fill="#DE8A3E"/><text x="691" y="115" font-size="13" fill="#FFFFFF" text-anchor="middle" font-weight="bold">3</text><text x="804.0" y="115" font-size="16.5" fill="#77400F" text-anchor="middle" font-weight="bold">Loss 계산</text><text x="790.0" y="135" font-size="12.5" fill="#77400F" text-anchor="middle" font-weight="normal" opacity="0.8">crossentropy</text><text x="790.0" y="158" font-size="13" fill="#77400F" text-anchor="middle" font-weight="normal">정답에서 벗어난 정도를</text><text x="790.0" y="177" font-size="13" fill="#77400F" text-anchor="middle" font-weight="normal">하나의 수치로 요약</text><rect x="665" y="292" width="250" height="128" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="691" cy="318" r="13" fill="#7B5EA7"/><text x="691" y="323" font-size="13" fill="#FFFFFF" text-anchor="middle" font-weight="bold">4</text><text x="804.0" y="323" font-size="16.5" fill="#3B295D" text-anchor="middle" font-weight="bold">역전파</text><text x="790.0" y="343" font-size="12.5" fill="#3B295D" text-anchor="middle" font-weight="normal" opacity="0.8">backpropagation</text><text x="790.0" y="366" font-size="13" fill="#3B295D" text-anchor="middle" font-weight="normal">어떤 내부 값이 Loss에</text><text x="790.0" y="385" font-size="13" fill="#3B295D" text-anchor="middle" font-weight="normal">얼마나 영향을 줬는지 계산</text><text x="790.0" y="408" font-size="12.5" fill="#3B295D" text-anchor="middle" font-weight="bold">→ 값은 아직 그대로입니다</text><rect x="345" y="292" width="250" height="128" rx="10" ry="10" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><circle cx="371" cy="318" r="13" fill="#4E9A57"/><text x="371" y="323" font-size="13" fill="#FFFFFF" text-anchor="middle" font-weight="bold">5</text><text x="484.0" y="323" font-size="16.5" fill="#1D4726" text-anchor="middle" font-weight="bold">Optimizer</text><text x="470.0" y="343" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="normal" opacity="0.8">Adam</text><text x="470.0" y="366" font-size="13" fill="#1D4726" text-anchor="middle" font-weight="normal">계산된 영향을 이용해</text><text x="470.0" y="385" font-size="13" fill="#1D4726" text-anchor="middle" font-weight="normal">내부 값을 실제로 수정</text><text x="470.0" y="408" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="bold">→ 여기서 값이 바뀝니다</text><rect x="25" y="292" width="250" height="128" rx="10" ry="10" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="51" cy="318" r="13" fill="#7B5EA7"/><text x="51" y="323" font-size="13" fill="#FFFFFF" text-anchor="middle" font-weight="bold">6</text><text x="164.0" y="323" font-size="16.5" fill="#3B295D" text-anchor="middle" font-weight="bold">다시 예측</text><text x="150.0" y="343" font-size="12.5" fill="#3B295D" text-anchor="middle" font-weight="normal" opacity="0.8">다음 batch</text><text x="150.0" y="366" font-size="13" fill="#3B295D" text-anchor="middle" font-weight="normal">수정된 값으로</text><text x="150.0" y="385" font-size="13" fill="#3B295D" text-anchor="middle" font-weight="normal">처음부터 반복</text><line x1="283" y1="148.0" x2="328" y2="148.0" stroke="#8A93A0" stroke-width="2"/><polygon points="337,148.0 327,142.0 327,154.0" fill="#8A93A0"/><line x1="603" y1="148.0" x2="648" y2="148.0" stroke="#8A93A0" stroke-width="2"/><polygon points="657,148.0 647,142.0 647,154.0" fill="#8A93A0"/><line x1="790.0" y1="216" x2="790.0" y2="279" stroke="#8A93A0" stroke-width="2"/><polygon points="790.0,288 784.0,278 796.0,278" fill="#8A93A0"/><line x1="657" y1="356.0" x2="612" y2="356.0" stroke="#8A93A0" stroke-width="2"/><polygon points="603,356.0 613,350.0 613,362.0" fill="#8A93A0"/><line x1="337" y1="356.0" x2="292" y2="356.0" stroke="#8A93A0" stroke-width="2"/><polygon points="283,356.0 293,350.0 293,362.0" fill="#8A93A0"/><line x1="150.0" y1="288" x2="150.0" y2="225" stroke="#8A93A0" stroke-width="2"/><polygon points="150.0,216 144.0,226 156.0,226" fill="#8A93A0"/><text x="180.0" y="256" font-size="13" fill="#8B93A0" text-anchor="start" font-weight="normal">반복</text><rect x="25" y="452" width="890" height="82" rx="10" ry="10" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="470.0" y="482" font-size="14.5" fill="#3C4552" text-anchor="middle" font-weight="normal">④ 역전파는 “영향을 계산하는” 단계이고,  ⑤ Optimizer는 “값을 실제로 바꾸는” 단계입니다.</text><text x="470.0" y="510" font-size="13.5" fill="#5A6270" text-anchor="middle" font-weight="normal">이 한 바퀴가 batch 마다 일어나고, 학습 데이터를 한 번 다 보면 1 epoch 이 끝납니다.</text></svg>

**이 한 바퀴가 계속 반복되는 것이 딥러닝의 학습입니다.**
아래에서 ③ Loss, ④ 역전파, ⑤ Optimizer 를 하나씩 봅니다.

---
## 06-2. Loss 는 무엇인가요?

> **모델의 예측이 정답에서 얼마나 벗어났는지를 하나의 숫자로 나타낸 값입니다.**

- 예측이 정답에 가까울수록 Loss 는 **작아집니다.**
- 예측이 정답에서 멀수록 Loss 는 **커집니다.**
- **학습은 이 값을 줄이는 방향으로 진행됩니다.**

그래서 학습이 잘 되고 있는지 확인하는 가장 간단한 방법은
**Loss 가 줄어들고 있는지 보는 것**입니다.

---
## 06-3. Backpropagation (역전파) 은 무엇을 하나요?

> **어떤 학습 가능한 값이 Loss 에 얼마나 영향을 주었는지를 계산합니다.**

- 역전파는 **"누가 얼마나 잘못했는지" 를 알아내는 단계**입니다.
- 즉, **어떻게 바꿔야 할지를 계산**합니다.
- **이 단계에서는 아직 값이 바뀌지 않습니다.**

---
## 06-4. Optimizer 는 무엇을 하나요?

> **역전파가 계산한 정보를 이용해, 학습 가능한 값을 실제로 수정합니다.**

- Optimizer 는 **"그러면 얼마만큼 고칠지" 를 실행하는 단계**입니다.
- **여기서 비로소 값이 바뀝니다.**

### 두 단계를 절대 같은 것으로 이해하지 마세요

| 단계 | 하는 일 | 값이 바뀌나요? |
|---|---|---|
| **역전파 (Backpropagation)** | 어떤 값이 Loss 에 **얼마나 영향을 주었는지 계산** | **아니요.** 계산만 합니다 |
| **Optimizer (옵티마이저)** | 그 계산 결과로 값을 **실제로 수정** | **네.** 여기서 값이 바뀝니다 |

### 학습을 시작하기 전에 정할 두 가지 숫자

| 설정 | 한 줄 설명 |
|---|---|
| **EPOCHS (반복 횟수)** | 전체 학습 데이터를 몇 번 다시 볼 것인가 |
| **LEARNING_RATE (학습률)** | 한 번 수정할 때 얼마나 크게 움직일 것인가 |

In [6]:
# ============================================================
# 학습 방법을 모델에게 알려 줍니다. (아직 학습은 하지 않습니다)
#
#   무엇을 줄일지      → loss       (Loss 를 줄이는 것이 목표)
#   어떻게 고칠지      → optimizer  (학습 가능한 내부 값을 실제로 수정)
#   얼마나 크게 고칠지 → learning_rate
#
# 오늘 확인할 것은 "Loss 가 줄어드는가" 하나뿐이므로
# 별도의 평가 지표는 넣지 않습니다. 학습 출력도 loss 중심으로 단순해집니다.
# ============================================================
EPOCHS = 20
LEARNING_RATE = 0.01

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
)

print("학습 준비가 끝났습니다.")

학습 준비가 끝났습니다.


---
## 06-5. 실제 학습 — `model.fit()`

앞에서 본 한 바퀴를 **실제로 돌립니다.**
`model.fit()` **한 줄**이 그 반복을 전부 대신해 줍니다.

**예측 → Loss → 역전파 → Optimizer → 파라미터 수정 → 다시 예측** ↺

### 무엇을 확인하면 되나요?

출력되는 숫자 중 **`loss` 가 epoch 이 지날수록 줄어드는지**만 보면 됩니다.
**Loss 가 줄어든다는 것은 학습 가능한 내부 값이 실제로 수정되고 있다**는 뜻입니다.

| 출력되는 이름 | 뜻 |
|---|---|
| `loss` | 학습에 사용한 문장들에 대한 Loss |
| `val_loss` | 학습에 쓰지 않고 따로 떼어 둔 문장들에 대한 Loss |

In [7]:
# ============================================================
# ★ 실제 학습 시작
#
# 입력:
#   X → 민원 문장
#   y → 각 민원의 정답 카테고리 번호
#
# model.fit() 내부에서 반복되는 일:
#
#   ① 현재 학습 파라미터로 예측
#   ② 정답과 비교
#   ③ Loss 계산
#   ④ 역전파로 어떤 값이 Loss 에 영향을 줬는지 계산
#   ⑤ Optimizer 가 학습 가능한 내부 값을 실제로 수정
#   ⑥ 수정된 값으로 다시 예측
#
# 이 과정을 여러 번 반복하며 Loss 를 줄여 갑니다.
# ============================================================

model.fit(
    X, y,                   # 민원 문장과 정답을 함께 보여 줍니다.
    epochs=EPOCHS,          # 전체 학습 데이터를 여러 번 반복합니다.
    batch_size=8,           # 8문장씩 보고 한 번씩 값을 수정합니다.
    validation_split=0.2,   # 일부 문장을 학습 중 확인용으로 따로 둡니다.
    verbose=2,              # 학습 진행 상황을 간단히 출력합니다.
)

print("학습이 끝났습니다.")

Epoch 1/20


6/6 - 1s - 226ms/step - loss: 1.0924 - val_loss: 1.0456


Epoch 2/20


6/6 - 0s - 15ms/step - loss: 1.0061 - val_loss: 0.9725


Epoch 3/20


...

Epoch 19/20


6/6 - 0s - 22ms/step - loss: 0.0034 - val_loss: 0.3614


Epoch 20/20


6/6 - 0s - 17ms/step - loss: 0.0030 - val_loss: 0.3613


학습이 끝났습니다.


---
---
# 07. `[결과 확인]`  학습 후 같은 문장 다시 보기

05장에서 **학습하기 전** 모델에 넣어 본 문장을 기억하시나요?
그때는 세 점수가 서로 비슷했습니다.
**똑같은 문장**을 학습이 끝난 모델에 다시 넣어 봅니다.

모델의 **코드도 구조도 하나도 바뀌지 않았습니다.** 바뀐 것은 내부 값뿐입니다.

In [8]:
# ============================================================
# [결과 확인] 05장과 똑같은 문장을, 학습이 끝난 모델에 다시 넣습니다.
#
# 모델 구조는 그대로이고, 학습으로 조정된 내부 값만 달라진 상태입니다.
# ============================================================
학습후점수 = model.predict(np.array([비교문장], dtype=object), verbose=0)[0]

# ------------------------------------------------------------
# [결과 보기용 보조 코드 — MLP 핵심 코드가 아닙니다]
# ------------------------------------------------------------
print(비교문장)
print(CLASS_NAMES)
print("학습 전 :", np.round(학습전점수, 3))
print("학습 후 :", np.round(학습후점수, 3))
print()

# np.argmax : 3개 카테고리 점수 중 가장 큰 값의 "위치(번호)" 를 찾습니다.
#
#   예) [0.02, 0.94, 0.04]
#       가장 큰 값은 두 번째 값
#       → 번호 1
#       → CLASS_NAMES[1] = "신고·납부"
print("모델이 제안하는 카테고리 :", CLASS_NAMES[int(np.argmax(학습후점수))])

소득금액증명서를 출력하고 싶습니다.


['증명서 발급', '신고·납부', '홈택스 이용']
학습 전 : [0.33  0.334 0.336]
학습 후 : [0.999 0.001 0.   ]



모델이 제안하는 카테고리 : 증명서 발급


<svg viewBox="0 0 940 440" role="img" aria-label="학습 전과 학습 후의 출력 점수 비교" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>학습 전과 학습 후의 출력 점수 비교</title><desc>같은 민원 문장에 대해 학습 전에는 세 카테고리 점수가 서로 비슷했으나 학습 후에는 증명서 발급 카테고리의 점수가 크게 높아진 것을 좌우로 비교한 막대그래프.</desc><rect x="0" y="0" width="940" height="440" fill="#FFFFFF"/><text x="470" y="32" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">학습 전과 학습 후, 같은 문장의 점수는 이렇게 달라집니다</text><text x="470" y="56" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">입력 문장 : “소득금액증명서를 출력하고 싶습니다.”  (노트북 실제 실행 결과)</text><rect x="30" y="80" width="400" height="272" rx="12" ry="12" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="230" y="110" font-size="17" fill="#555C68" text-anchor="middle" font-weight="bold">학습 전</text><text x="230" y="132" font-size="12.5" fill="#6B7280" text-anchor="middle" font-weight="normal">아직 학습하지 않은 초기값</text><text x="50" y="183" font-size="14" fill="#5A6270" text-anchor="start" font-weight="normal">증명서 발급</text><rect x="150" y="167" width="170" height="16" rx="4" fill="#E4E8EE"/><rect x="150" y="167" width="56.1" height="16" rx="4" fill="#B9C0CA"/><text x="332" y="183" font-size="14" fill="#5A6270" text-anchor="start" font-weight="normal">33.0%</text><text x="50" y="223" font-size="14" fill="#5A6270" text-anchor="start" font-weight="normal">신고·납부</text><rect x="150" y="207" width="170" height="16" rx="4" fill="#E4E8EE"/><rect x="150" y="207" width="56.8" height="16" rx="4" fill="#B9C0CA"/><text x="332" y="223" font-size="14" fill="#5A6270" text-anchor="start" font-weight="normal">33.4%</text><text x="50" y="263" font-size="14" fill="#5A6270" text-anchor="start" font-weight="normal">홈택스 이용</text><rect x="150" y="247" width="170" height="16" rx="4" fill="#E4E8EE"/><rect x="150" y="247" width="57.1" height="16" rx="4" fill="#B9C0CA"/><text x="332" y="263" font-size="14" fill="#5A6270" text-anchor="start" font-weight="normal">33.6%</text><text x="230" y="310" font-size="13" fill="#4B5563" text-anchor="middle" font-weight="normal">아직 유용한 패턴을 배우지 못해</text><text x="230" y="331" font-size="13" fill="#4B5563" text-anchor="middle" font-weight="normal">세 점수가 서로 비슷합니다.</text><rect x="510" y="80" width="400" height="272" rx="12" ry="12" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="710" y="110" font-size="17" fill="#1D4726" text-anchor="middle" font-weight="bold">학습 후</text><text x="710" y="132" font-size="12.5" fill="#6B7280" text-anchor="middle" font-weight="normal">20 epoch 학습 완료</text><text x="530" y="183" font-size="14" fill="#1D4726" text-anchor="start" font-weight="bold">증명서 발급</text><rect x="630" y="167" width="170" height="16" rx="4" fill="#E4E8EE"/><rect x="630" y="167" width="169.8" height="16" rx="4" fill="#4E9A57"/><text x="812" y="183" font-size="14" fill="#1D4726" text-anchor="start" font-weight="bold">99.9%</text><text x="530" y="223" font-size="14" fill="#5A6270" text-anchor="start" font-weight="normal">신고·납부</text><rect x="630" y="207" width="170" height="16" rx="4" fill="#E4E8EE"/><rect x="630" y="207" width="3.0" height="16" rx="4" fill="#B9C0CA"/><text x="812" y="223" font-size="14" fill="#5A6270" text-anchor="start" font-weight="normal">0.1%</text><text x="530" y="263" font-size="14" fill="#5A6270" text-anchor="start" font-weight="normal">홈택스 이용</text><rect x="630" y="247" width="170" height="16" rx="4" fill="#E4E8EE"/><rect x="630" y="247" width="3.0" height="16" rx="4" fill="#B9C0CA"/><text x="812" y="263" font-size="14" fill="#5A6270" text-anchor="start" font-weight="normal">0.0%</text><text x="710" y="310" font-size="13" fill="#4B5563" text-anchor="middle" font-weight="normal">학습 가능한 내부 값이 조정되면서</text><text x="710" y="331" font-size="13" fill="#4B5563" text-anchor="middle" font-weight="normal">출력 점수가 크게 달라졌습니다.</text><text x="470" y="194" font-size="13.5" fill="#6B7280" text-anchor="middle" font-weight="bold">학습</text><text x="470" y="240" font-size="12.5" fill="#9AA1AC" text-anchor="middle" font-weight="normal">model.fit()</text><line x1="434" y1="216" x2="499" y2="216" stroke="#8A93A0" stroke-width="2"/><polygon points="508,216 498,210 498,222" fill="#8A93A0"/><text x="470" y="388" font-size="14" fill="#3C4552" text-anchor="middle" font-weight="normal">달라진 것은 모델의 구조가 아니라, 학습으로 조정되는 내부 값입니다.</text><text x="470" y="412" font-size="13.5" fill="#6B7280" text-anchor="middle" font-weight="normal">Embedding 의 숫자 표현과 Dense 층의 Weight · Bias 가 함께 조정된 결과입니다.</text></svg>

### 여기서 반드시 이해할 것

> **모델의 구조가 바뀐 것이 아닙니다.**
>
> **학습을 통해 모델 내부의 학습 가능한 값이 바뀌었기 때문에 결과가 달라졌습니다.**

| | 학습 전 | 학습 후 |
|---|---|---|
| 모델 구조 (층 · 뉴런 수 · 코드) | 그대로 | **똑같이 그대로** |
| 학습 가능한 내부 값 | 아직 조정되지 않음 | **조정됨** |
| 같은 문장의 출력 점수 | 세 점수가 비슷 | **한쪽이 크게 높아짐** |

층의 개수도, 뉴런 수도, 코드도 **하나도 바뀌지 않았습니다.**
**학습은 층을 새로 추가하는 과정이 아니라, 내부 값을 조정하는 과정입니다.**

정확하게는 다음이 함께 조정됩니다.

- **Embedding 의 숫자 표현** (입력 준비 쪽)
- **Dense 층의 Weight / Bias** (MLP 쪽)

> 다만 **MLP 를 이해할 때는 Dense 층의 Weight/Bias 에 초점**을 맞춥니다.

---
---
# 08. `[결과 확인]`  새로운 민원 3개 분류 (추론)

> **📍 현재 위치**  
> 새 민원 → 학습된 모델 → **★ 추론 ← 지금 여기** → 3개 점수 → 가장 높은 후보 → 담당자 최종 확인

**학습에 사용하지 않은 새로운 민원 문장** 3개를 모델에 넣어 봅니다.
세 문장은 각각 다른 카테고리에 해당합니다.

| 단계 | 내용 |
|---|---|
| 입력 | 새 민원 문장 |
| 처리 | 학습된 모델이 Forward 만 수행 |
| 출력 | 3개 카테고리 점수 |
| 판단 | 가장 큰 점수의 위치 → 후보 카테고리 |
| 최종 | **담당자가 확인** |

> ### 직접 해 보기 — **문장만 바꾸세요.**
> 아래 셀에서 **따옴표 안의 문장만** 바꾸고 다시 실행해 보세요.
> **모델 코드는 건드리지 않습니다.**
> 단, **실제 민원 내용이나 개인정보는 넣지 않습니다.**

In [9]:
# ============================================================
# ★ 여기서는 아래 문장만 바꾸면 됩니다.
# ============================================================
새민원 = np.array([
    "사업자등록증명원이 급하게 필요합니다.",
    "종합소득세를 인터넷으로 신고하려면 어떻게 하나요?",
    "홈택스에 접속하려는데 로그인이 계속 실패합니다.",
], dtype=object)
# ============================================================

# ============================================================
# [추론]
#
# 학습이 끝난 모델의 내부 값은 더 이상 수정하지 않습니다.
#
# 새 민원을 넣으면:
#
#   ① TextVectorization
#   ② Embedding
#   ③ Pooling
#   ④ Dense + ReLU
#   ⑤ Dense + Softmax
#
# 순서로 Forward 만 수행하여 3개 카테고리 점수를 계산합니다.
# 정답 비교도, Loss 도, 역전파도, 값 수정도 일어나지 않습니다.
# ============================================================
점수표 = model.predict(
    새민원,
    verbose=0,
)

# ------------------------------------------------------------
# [결과 보기용 보조 코드 — MLP 핵심 코드가 아닙니다]
# 문장을 하나씩 꺼내 점수와 함께 보기 좋게 출력합니다.
# ------------------------------------------------------------
print(CLASS_NAMES)
print()
for 문장, 점수 in zip(새민원, 점수표):
    # np.argmax : 3개 카테고리 점수 중 가장 큰 값의 위치를 찾습니다.
    #
    #   예) [0.02, 0.94, 0.04]
    #       가장 큰 값은 두 번째 값 → 번호 1 → "신고·납부"
    예측번호 = int(np.argmax(점수))

    print(문장)
    print("   →", CLASS_NAMES[예측번호], np.round(점수, 3))
    print()

['증명서 발급', '신고·납부', '홈택스 이용']



사업자등록증명원이 급하게 필요합니다.
   → 증명서 발급 [0.996 0.003 0.001]

종합소득세를 인터넷으로 신고하려면 어떻게 하나요?
   → 신고·납부 [0.224 0.772 0.004]

홈택스에 접속하려는데 로그인이 계속 실패합니다.
   → 홈택스 이용 [0.    0.003 0.997]



### 결과에서 확인할 점

- 세 점수의 **합은 항상 1** 입니다. (Softmax)
- 세 문장이 **서로 다른 카테고리**로 분류됩니다.
- 점수가 **한쪽으로 확실히 몰린 문장**과 **비교적 나뉘는 문장**이 있습니다.
  점수가 나뉘는 문장은 **담당자의 확인이 특히 필요한 사례**입니다.

> ### 모델이 문장의 "의미를 이해한" 것은 아닙니다
> 이 모델은 학습 데이터에서 **반복적으로 나타난 글자·표현 패턴**을 이용해 카테고리를 구분합니다.
> 사람처럼 문맥이나 민원의 의도를 이해하는 것이 아닙니다.
> 그래서 **최종 판단과 책임은 담당자에게 있습니다.**

---
---
# 09. 학습과 추론은 무엇이 다를까?

<svg viewBox="0 0 940 640" role="img" aria-label="학습과 추론의 차이" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>학습과 추론의 차이</title><desc>학습은 입력, 예측, 정답 비교, Loss 계산, 역전파, Weight 수정의 여섯 단계를 거쳐 Weight와 Bias가 바뀌지만, 추론은 새 입력, 계산, 예측 결과의 세 단계만 거치며 Weight와 Bias가 고정된다는 것을 좌우로 비교한 그림</desc><rect x="0" y="0" width="940" height="640" fill="#FFFFFF"/><text x="470.0" y="32" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">학습과 추론은 무엇이 다를까요?</text><text x="470.0" y="56" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">같은 모델이지만 수행하는 단계가 다릅니다</text><rect x="30" y="80" width="380" height="520" rx="12" ry="12" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><rect x="530" y="80" width="380" height="520" rx="12" ry="12" fill="#F5F6F8" stroke="#CBD0D8" stroke-width="1.5"/><text x="220" y="108" font-size="17" fill="#1F2733" text-anchor="middle" font-weight="bold">학습 (Training)</text><text x="220" y="128" font-size="12.5" fill="#6B7280" text-anchor="middle" font-weight="normal">model.fit()</text><text x="720" y="108" font-size="17" fill="#1F2733" text-anchor="middle" font-weight="bold">추론 (Inference)</text><text x="720" y="128" font-size="12.5" fill="#6B7280" text-anchor="middle" font-weight="normal">model.predict()</text><rect x="50" y="148" width="340" height="44" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="220" y="176" font-size="15.5" fill="#1B3A5E" text-anchor="middle" font-weight="bold">입력</text><line x1="220" y1="192" x2="220" y2="205" stroke="#8A93A0" stroke-width="2"/><polygon points="220,214 214,204 226,204" fill="#8A93A0"/><rect x="50" y="214" width="340" height="44" rx="8" ry="8" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="220" y="242" font-size="15.5" fill="#3B295D" text-anchor="middle" font-weight="bold">예측</text><line x1="220" y1="258" x2="220" y2="271" stroke="#8A93A0" stroke-width="2"/><polygon points="220,280 214,270 226,270" fill="#8A93A0"/><rect x="50" y="280" width="340" height="44" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="220" y="308" font-size="15.5" fill="#1B3A5E" text-anchor="middle" font-weight="bold">정답과 비교</text><line x1="220" y1="324" x2="220" y2="337" stroke="#8A93A0" stroke-width="2"/><polygon points="220,346 214,336 226,336" fill="#8A93A0"/><rect x="50" y="346" width="340" height="44" rx="8" ry="8" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="220" y="374" font-size="15.5" fill="#77400F" text-anchor="middle" font-weight="bold">Loss 계산</text><line x1="220" y1="390" x2="220" y2="403" stroke="#8A93A0" stroke-width="2"/><polygon points="220,412 214,402 226,402" fill="#8A93A0"/><rect x="50" y="412" width="340" height="44" rx="8" ry="8" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="220" y="440" font-size="15.5" fill="#3B295D" text-anchor="middle" font-weight="bold">역전파</text><line x1="220" y1="456" x2="220" y2="469" stroke="#8A93A0" stroke-width="2"/><polygon points="220,478 214,468 226,468" fill="#8A93A0"/><rect x="50" y="478" width="340" height="44" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="220" y="506" font-size="15.5" fill="#1D4726" text-anchor="middle" font-weight="bold">Weight · Bias 수정</text><rect x="550" y="148" width="340" height="44" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="720" y="176" font-size="15.5" fill="#1B3A5E" text-anchor="middle" font-weight="bold">새 민원 문장 입력</text><line x1="720" y1="192" x2="720" y2="205" stroke="#8A93A0" stroke-width="2"/><polygon points="720,214 714,204 726,204" fill="#8A93A0"/><rect x="550" y="214" width="340" height="44" rx="8" ry="8" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><text x="720" y="242" font-size="15.5" fill="#3B295D" text-anchor="middle" font-weight="bold">계산</text><line x1="720" y1="258" x2="720" y2="271" stroke="#8A93A0" stroke-width="2"/><polygon points="720,280 714,270 726,270" fill="#8A93A0"/><rect x="550" y="280" width="340" height="44" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="720" y="308" font-size="15.5" fill="#1D4726" text-anchor="middle" font-weight="bold">예측 결과</text><rect x="550" y="356" width="340" height="40" rx="8" ry="8" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="720" y="382" font-size="14" fill="#77400F" text-anchor="middle" font-weight="normal">✕  정답과 비교하지 않습니다</text><rect x="550" y="408" width="340" height="40" rx="8" ry="8" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="720" y="434" font-size="14" fill="#77400F" text-anchor="middle" font-weight="normal">✕  Loss 계산과 역전파가 없습니다</text><rect x="550" y="460" width="340" height="40" rx="8" ry="8" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="720" y="486" font-size="14" fill="#77400F" text-anchor="middle" font-weight="normal">✕  Weight · Bias를 수정하지 않습니다</text><rect x="50" y="538" width="340" height="46" rx="8" fill="#4E9A57"/><text x="220" y="567" font-size="15.5" fill="#FFFFFF" text-anchor="middle" font-weight="bold">Weight와 Bias가 바뀝니다</text><rect x="550" y="538" width="340" height="46" rx="8" fill="#4C78A8"/><text x="720" y="567" font-size="15.5" fill="#FFFFFF" text-anchor="middle" font-weight="bold">Weight와 Bias는 고정됩니다</text><text x="470" y="300" font-size="15" fill="#3C4552" text-anchor="middle" font-weight="bold">같은 모델</text><text x="470" y="326" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">하지만</text><text x="470" y="346" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">수행 과정은</text><text x="470" y="366" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">다릅니다</text></svg>

> **학습**(`model.fit`)에서는 **정답을 사용하여 내부 값을 바꿉니다.**
>
> **추론**(`model.predict`)에서는 **이미 학습된 내부 값을 바꾸지 않고,
> 새로운 입력의 점수만 계산합니다.**

`model.predict()` 에서는 **정답 비교도, Loss 계산도, 역전파도, 파라미터 수정도 일어나지 않습니다.**
이미 학습으로 정해진 값을 그대로 사용해 **계산만** 합니다.

---
---
# 10. 핵심 정리

## 10-1. 전체 알고리즘 한눈에 보기

처음에 본 지도를 마지막에 한 번 더 봅니다.
이제는 각 상자 안에서 무슨 일이 일어나는지 설명할 수 있어야 합니다.

<svg viewBox="0 0 940 664" role="img" aria-label="노트북 전체 알고리즘 한 장" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:900px;height:auto;display:block;margin:0 auto;font-family:'Malgun Gothic','Noto Sans KR','NanumGothic','Apple SD Gothic Neo',sans-serif;"><title>민원 분류 프로그램 전체 흐름</title><desc>민원 문장이 입력 준비 단계인 TextVectorization, Embedding, GlobalAveragePooling1D 를 거쳐 숫자 16개가 되고, MLP 핵심인 Dense 16 과 ReLU, Dense 3 과 Softmax 를 지나 3개 카테고리 점수가 되는 계산 흐름과, 그 점수를 정답과 비교해 Loss 를 구하고 역전파로 영향을 계산한 뒤 Optimizer 가 학습 파라미터를 실제로 수정하는 학습 반복 과정, 그리고 학습이 끝난 모델이 새 민원의 후보 카테고리를 제안하고 담당자가 최종 확인하는 실제 사용 과정을 다섯 구역으로 나누어 보여 주는 전체 지도.</desc><rect x="0" y="0" width="940" height="664" fill="#FFFFFF"/><text x="470" y="32" font-size="21" fill="#1F2733" text-anchor="middle" font-weight="bold">이 노트북 전체가 하는 일 한 장으로 보기</text><text x="470" y="56" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">회색 = 입력 준비 (MLP 아님)   ·   파랑 = ★ MLP 핵심   ·   주황 = ★ 학습</text><rect x="24" y="76" width="300" height="272" rx="14" ry="14" fill="#F7F8FA" stroke="#D7DBE2" stroke-width="1.5"/><text x="44" y="102" font-size="14" fill="#6B7280" text-anchor="start" font-weight="bold">① 입력 준비</text><text x="44" y="120" font-size="11.5" fill="#8B93A0" text-anchor="start" font-weight="normal">MLP 자체는 아닙니다</text><rect x="44" y="132" width="260" height="36" rx="8" ry="8" fill="#EDEFF3" stroke="#AEB6C2" stroke-width="1.5"/><text x="174" y="149" font-size="13" fill="#3D4552" text-anchor="middle" font-weight="bold">민원 문장</text><text x="174" y="162" font-size="9.5" fill="#6B7280" text-anchor="middle" font-weight="normal">한국어 한 줄</text><line x1="174" y1="168" x2="174" y2="173" stroke="#8A93A0" stroke-width="2"/><polygon points="174,182 168,172 180,172" fill="#8A93A0"/><rect x="44" y="182" width="260" height="36" rx="8" ry="8" fill="#EDEFF3" stroke="#AEB6C2" stroke-width="1.5"/><text x="174" y="199" font-size="13" fill="#3D4552" text-anchor="middle" font-weight="bold">TextVectorization</text><text x="174" y="212" font-size="9.5" fill="#6B7280" text-anchor="middle" font-weight="normal">글자 토큰 → 토큰 ID</text><line x1="174" y1="218" x2="174" y2="223" stroke="#8A93A0" stroke-width="2"/><polygon points="174,232 168,222 180,222" fill="#8A93A0"/><rect x="44" y="232" width="260" height="36" rx="8" ry="8" fill="#EDEFF3" stroke="#AEB6C2" stroke-width="1.5"/><text x="174" y="249" font-size="13" fill="#3D4552" text-anchor="middle" font-weight="bold">Embedding</text><text x="174" y="262" font-size="9.5" fill="#6B7280" text-anchor="middle" font-weight="normal">토큰 ID → 숫자 16개</text><line x1="174" y1="268" x2="174" y2="273" stroke="#8A93A0" stroke-width="2"/><polygon points="174,282 168,272 180,272" fill="#8A93A0"/><rect x="44" y="282" width="260" height="36" rx="8" ry="8" fill="#EDEFF3" stroke="#AEB6C2" stroke-width="1.5"/><text x="174" y="299" font-size="13" fill="#3D4552" text-anchor="middle" font-weight="bold">GlobalAveragePooling1D</text><text x="174" y="312" font-size="9.5" fill="#6B7280" text-anchor="middle" font-weight="normal">평균 → 문장 숫자 16개</text><text x="174" y="336" font-size="12.5" fill="#77400F" text-anchor="middle" font-weight="bold">→ 문장 하나 = 숫자 16개</text><rect x="356" y="76" width="300" height="272" rx="14" ry="14" fill="#F2F6FC" stroke="#4C78A8" stroke-width="2.5"/><text x="376" y="102" font-size="14" fill="#2F5C93" text-anchor="start" font-weight="bold">② ★ MLP 핵심</text><text x="376" y="120" font-size="11.5" fill="#4C78A8" text-anchor="start" font-weight="normal">오늘 이해해야 하는 부분</text><rect x="376" y="132" width="260" height="26" rx="8" ry="8" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="506" y="150" font-size="13" fill="#77400F" text-anchor="middle" font-weight="bold">입력 : 숫자 16개</text><line x1="506" y1="158" x2="506" y2="163" stroke="#8A93A0" stroke-width="2"/><polygon points="506,172 500,162 512,162" fill="#8A93A0"/><rect x="376" y="172" width="260" height="42" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="2"/><text x="506" y="190" font-size="14" fill="#1B3A5E" text-anchor="middle" font-weight="bold">Dense(16)</text><text x="506" y="206" font-size="11" fill="#1B3A5E" text-anchor="middle" font-weight="normal">Weight × Input + Bias</text><line x1="506" y1="214" x2="506" y2="219" stroke="#8A93A0" stroke-width="2"/><polygon points="506,228 500,218 512,218" fill="#8A93A0"/><rect x="376" y="228" width="260" height="24" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="506" y="245" font-size="13" fill="#0E4A46" text-anchor="middle" font-weight="bold">ReLU</text><line x1="506" y1="252" x2="506" y2="257" stroke="#8A93A0" stroke-width="2"/><polygon points="506,266 500,256 512,256" fill="#8A93A0"/><rect x="376" y="266" width="260" height="42" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="2"/><text x="506" y="284" font-size="14" fill="#1B3A5E" text-anchor="middle" font-weight="bold">Dense(3)</text><text x="506" y="300" font-size="11" fill="#1B3A5E" text-anchor="middle" font-weight="normal">Weight × Hidden + Bias</text><line x1="506" y1="308" x2="506" y2="313" stroke="#8A93A0" stroke-width="2"/><polygon points="506,322 500,312 512,312" fill="#8A93A0"/><rect x="376" y="322" width="260" height="24" rx="8" ry="8" fill="#E1F3F2" stroke="#2E9E96" stroke-width="1.5"/><text x="506" y="339" font-size="13" fill="#0E4A46" text-anchor="middle" font-weight="bold">Softmax</text><rect x="688" y="76" width="228" height="272" rx="14" ry="14" fill="#F7F8FA" stroke="#D7DBE2" stroke-width="1.5"/><text x="708" y="102" font-size="14" fill="#1D4726" text-anchor="start" font-weight="bold">③ 결과</text><rect x="708" y="132" width="188" height="44" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="2"/><text x="802" y="160" font-size="14" fill="#1D4726" text-anchor="middle" font-weight="bold">3개 카테고리 점수</text><rect x="708" y="192" width="188" height="24" rx="6" ry="6" fill="#FFFFFF" stroke="#4E9A57" stroke-width="1.2"/><text x="802" y="209" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="normal">증명서 발급</text><rect x="708" y="220" width="188" height="24" rx="6" ry="6" fill="#FFFFFF" stroke="#4E9A57" stroke-width="1.2"/><text x="802" y="237" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="normal">신고·납부</text><rect x="708" y="248" width="188" height="24" rx="6" ry="6" fill="#FFFFFF" stroke="#4E9A57" stroke-width="1.2"/><text x="802" y="265" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="normal">홈택스 이용</text><line x1="802" y1="272" x2="802" y2="281" stroke="#8A93A0" stroke-width="2"/><polygon points="802,290 796,280 808,280" fill="#8A93A0"/><rect x="708" y="290" width="188" height="44" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="802" y="308" font-size="12" fill="#1D4726" text-anchor="middle" font-weight="normal">가장 높은 카테고리를</text><text x="802" y="325" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="bold">후보로 제안</text><line x1="328" y1="212" x2="345" y2="212" stroke="#8A93A0" stroke-width="2"/><polygon points="354,212 344,206 344,218" fill="#8A93A0"/><line x1="660" y1="212" x2="677" y2="212" stroke="#8A93A0" stroke-width="2"/><polygon points="686,212 676,206 676,218" fill="#8A93A0"/><rect x="24" y="386" width="560" height="210" rx="14" ry="14" fill="#FDF6EE" stroke="#DE8A3E" stroke-width="2.5"/><text x="44" y="412" font-size="14" fill="#77400F" text-anchor="start" font-weight="bold">④ ★ 학습  —  model.fit()</text><rect x="44" y="424" width="92" height="68" rx="8" ry="8" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="59" cy="439" r="11" fill="#7B5EA7"/><text x="59" y="444" font-size="12" fill="#FFFFFF" text-anchor="middle" font-weight="bold">1</text><text x="90" y="468" font-size="13" fill="#3B295D" text-anchor="middle" font-weight="bold">예측</text><text x="90" y="484" font-size="10.5" fill="#3B295D" text-anchor="middle" font-weight="normal">지금 값으로 계산</text><line x1="138" y1="458" x2="141" y2="458" stroke="#8A93A0" stroke-width="2"/><polygon points="150,458 140,452 140,464" fill="#8A93A0"/><rect x="151" y="424" width="92" height="68" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><circle cx="166" cy="439" r="11" fill="#4C78A8"/><text x="166" y="444" font-size="12" fill="#FFFFFF" text-anchor="middle" font-weight="bold">2</text><text x="197" y="468" font-size="13" fill="#1B3A5E" text-anchor="middle" font-weight="bold">정답과 비교</text><text x="197" y="484" font-size="10.5" fill="#1B3A5E" text-anchor="middle" font-weight="normal">Loss 계산</text><line x1="245" y1="458" x2="248" y2="458" stroke="#8A93A0" stroke-width="2"/><polygon points="257,458 247,452 247,464" fill="#8A93A0"/><rect x="258" y="424" width="92" height="68" rx="8" ry="8" fill="#EEE9F7" stroke="#7B5EA7" stroke-width="1.5"/><circle cx="273" cy="439" r="11" fill="#7B5EA7"/><text x="273" y="444" font-size="12" fill="#FFFFFF" text-anchor="middle" font-weight="bold">3</text><text x="304" y="468" font-size="13" fill="#3B295D" text-anchor="middle" font-weight="bold">역전파</text><text x="304" y="484" font-size="10.5" fill="#3B295D" text-anchor="middle" font-weight="normal">영향만 계산</text><line x1="352" y1="458" x2="355" y2="458" stroke="#8A93A0" stroke-width="2"/><polygon points="364,458 354,452 354,464" fill="#8A93A0"/><rect x="365" y="424" width="92" height="68" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><circle cx="380" cy="439" r="11" fill="#4E9A57"/><text x="380" y="444" font-size="12" fill="#FFFFFF" text-anchor="middle" font-weight="bold">4</text><text x="411" y="468" font-size="13" fill="#1D4726" text-anchor="middle" font-weight="bold">Optimizer</text><text x="411" y="484" font-size="10.5" fill="#1D4726" text-anchor="middle" font-weight="normal">값을 실제 수정</text><line x1="459" y1="458" x2="462" y2="458" stroke="#8A93A0" stroke-width="2"/><polygon points="471,458 461,452 461,464" fill="#8A93A0"/><rect x="472" y="424" width="92" height="68" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><circle cx="487" cy="439" r="11" fill="#4E9A57"/><text x="487" y="444" font-size="12" fill="#FFFFFF" text-anchor="middle" font-weight="bold">5</text><text x="518" y="468" font-size="13" fill="#1D4726" text-anchor="middle" font-weight="bold">파라미터</text><text x="518" y="484" font-size="10.5" fill="#1D4726" text-anchor="middle" font-weight="normal">수정 완료</text><line x1="518" y1="492" x2="518" y2="540" stroke="#8A93A0" stroke-width="2"/><line x1="518" y1="540" x2="90" y2="540" stroke="#8A93A0" stroke-width="2"/><line x1="90" y1="540" x2="90" y2="501" stroke="#8A93A0" stroke-width="2"/><polygon points="90,492 84,502 96,502" fill="#8A93A0"/><text x="304" y="532" font-size="12" fill="#77400F" text-anchor="middle" font-weight="bold">↺ 이 한 바퀴가 계속 반복됩니다</text><rect x="616" y="386" width="300" height="210" rx="14" ry="14" fill="#F7F8FA" stroke="#D7DBE2" stroke-width="1.5"/><text x="636" y="412" font-size="13.5" fill="#3D4552" text-anchor="start" font-weight="bold">⑤ 실제 사용 (학습이 끝난 뒤)</text><rect x="636" y="424" width="260" height="28" rx="8" ry="8" fill="#E8F0FB" stroke="#4C78A8" stroke-width="1.5"/><text x="766" y="443" font-size="12.5" fill="#1B3A5E" text-anchor="middle" font-weight="bold">새 민원 문장</text><line x1="766" y1="452" x2="766" y2="457" stroke="#8A93A0" stroke-width="2"/><polygon points="766,466 760,456 772,456" fill="#8A93A0"/><rect x="636" y="466" width="260" height="28" rx="8" ry="8" fill="#EDEFF3" stroke="#AEB6C2" stroke-width="1.5"/><text x="766" y="485" font-size="12.5" fill="#3D4552" text-anchor="middle" font-weight="bold">학습된 모델 — 계산만</text><line x1="766" y1="494" x2="766" y2="499" stroke="#8A93A0" stroke-width="2"/><polygon points="766,508 760,498 772,498" fill="#8A93A0"/><rect x="636" y="508" width="260" height="28" rx="8" ry="8" fill="#E7F3E9" stroke="#4E9A57" stroke-width="1.5"/><text x="766" y="527" font-size="12.5" fill="#1D4726" text-anchor="middle" font-weight="bold">3개 점수 → 가장 높은 후보</text><line x1="766" y1="536" x2="766" y2="541" stroke="#8A93A0" stroke-width="2"/><polygon points="766,550 760,540 772,540" fill="#8A93A0"/><rect x="636" y="550" width="260" height="28" rx="8" ry="8" fill="#FCF0E4" stroke="#DE8A3E" stroke-width="1.5"/><text x="766" y="569" font-size="12.5" fill="#77400F" text-anchor="middle" font-weight="bold">담당자 최종 확인</text><line x1="802" y1="348" x2="802" y2="368" stroke="#8A93A0" stroke-width="2" stroke-dasharray="5 4"/><line x1="802" y1="368" x2="304" y2="368" stroke="#8A93A0" stroke-width="2" stroke-dasharray="5 4"/><line x1="304" y1="368" x2="304" y2="377" stroke="#8A93A0" stroke-width="2"/><polygon points="304,386 298,376 310,376" fill="#8A93A0"/><rect x="382" y="354" width="342" height="20" fill="#FFFFFF"/><text x="553" y="369" font-size="12" fill="#77400F" text-anchor="middle" font-weight="bold">학습할 때는 이 점수를 정답과 비교합니다</text><line x1="588" y1="491" x2="605" y2="491" stroke="#8A93A0" stroke-width="2"/><polygon points="614,491 604,485 604,497" fill="#8A93A0"/><text x="470" y="624" font-size="13.5" fill="#3C4552" text-anchor="middle" font-weight="normal">① 입력 준비는 MLP 가 아닙니다. MLP 는 숫자 16개를 받는 ② 부터 시작합니다.</text><text x="470" y="648" font-size="13" fill="#6B7280" text-anchor="middle" font-weight="normal">④ 학습은 모델의 구조를 바꾸는 것이 아니라, 모델 내부의 학습 가능한 값을 조정하는 과정입니다.</text></svg>

| 구간 | 무엇이 들어가나 | MLP 인가? |
|---|---|---|
| **입력 준비** | `TextVectorization` · `Embedding` · `Pooling` | **아니요** |
| **★ MLP 핵심** | `Dense` · `Weight/Bias` · `ReLU` · `Dense` · `Softmax` | **네** |
| **★ 학습 핵심** | `Loss` · `Backpropagation` · `Optimizer` · `Parameter Update` | 학습 과정 |

---
## 10-2. 내가 이해했는지 확인하기

노트북을 덮기 전에, 아래를 **자기 말로** 말할 수 있는지 확인해 보세요.

### 입력 준비

1. 이번 실습에서는 문장을 무엇 단위로 나누나요? (**글자 토큰**)
2. 토큰 ID 란 무엇인가요? (토큰을 구분하기 위한 **번호**)
3. 토큰 ID 의 숫자 크기에 의미가 있나요? (**없습니다**)
4. Embedding 은 단어만 처리하나요? (**아니요.** 토큰 ID 를 숫자 벡터로 바꿉니다)
5. Embedding 을 거치면 글자 하나는 숫자 몇 개가 되나요? (**16개**)
6. Pooling 후 문장 하나는 왜 숫자 16개인가요? (**같은 자리끼리 평균**)

### MLP

7. 어디부터가 MLP 인가요? (**숫자 16개를 받는 Dense 부터**)
8. 뉴런 하나는 무엇을 계산하나요? (**입력 × Weight 를 더하고 Bias 를 더한 뒤 활성화 함수**)
9. Weight 는 무엇인가요? (각 입력이 **얼마나, 어느 방향으로** 영향을 줄지 조정하는 값)
10. Bias 는 무엇인가요? (더한 결과 전체를 **위/아래로 이동**시키는 값)
11. ReLU 는 무엇을 하나요? (**음수는 0, 양수는 그대로**)
12. `Dense(16)` 의 출력은 왜 16개인가요? (**뉴런이 16개**이기 때문)
13. `Dense(3)` 은 왜 3개인가요? (**카테고리가 3개**이기 때문)
14. Softmax 는 무엇을 하나요? (세 값을 **비교하기 쉬운 형태**로)

### 학습과 추론

15. Loss 는 무엇인가요? (**정답에서 벗어난 정도**를 하나의 숫자로)
16. 역전파와 Optimizer 는 무엇이 다른가요? (**계산** vs **실제 수정**)
17. 학습 전후에 모델 구조가 달라지나요? (**아니요. 내부 값만 달라집니다**)
18. 추론에서는 왜 내부 값이 바뀌지 않나요? (**정답 비교 · Loss · 역전파를 하지 않기 때문**)

막히는 항목이 있으면 해당 장으로 돌아가면 됩니다.

---
## 10-3. 이번 실습의 범위와 한계

> 이번 실습은 딥러닝의 기본 구성 요소인 **Dense 기반 MLP 의 학습 원리**를
> 아주 작은 모델로 확인한 것입니다.
> 이 모델 하나가 모든 딥러닝 모델의 구조를 대표하지는 않습니다.

| 이번 실습의 선택 | 그래서 생기는 한계 |
|---|---|
| **글자 단위 토큰** | 설명은 쉬워지지만, 낱말 수준의 의미를 직접 다루지는 않습니다 |
| **GlobalAveragePooling1D** | 글자별 정보를 평균으로 합치므로 **순서 정보가 남지 않습니다** |
| **단순 MLP (Dense 2층)** | 문장 안의 관계를 층층이 해석하는 구조가 아닙니다 |
| **성능 평가를 하지 않음** | 이 모델의 정확도를 주장하지 않습니다 |

> **이번 모델은 글자 표현을 평균 내므로, 문장의 세밀한 순서나 긴 문맥 관계를 충분히 활용하지 못합니다.**

| | 다루는 방식 |
|---|---|
| **오늘 — MLP 기반 단순 모델** | 어떤 글자와 짧은 표현 패턴이 문장에 포함되어 있는지를 중심으로 구분 |
| **다음 — CNN / RNN / Attention / Transformer** | 순서와 문맥 관계를 더 본격적으로 처리 |

## 마지막으로 기억할 것

1. **AI 는 담당 분야를 제안하는 단계까지 수행하고, 최종 판단과 책임은 담당자에게 있습니다.**
2. 실제 업무에 적용할 때는 **개인정보와 내부 비공개 자료를 입력하지 않습니다.**